# Is This AI Slop? (ITAIS) — train the LFM2.5 encoder

**Runtime → Change runtime type → T4 GPU.** Then **Runtime → Run all**.

Trains one model that does both jobs:

1. **Classify** the text: slop or not slop.
2. **Why:** which sentences + spans, with named patterns.

Base: `LiquidAI/LFM2.5-Encoder-350M` (bidirectional masked-LM encoder, ~354M params),
fine-tuned 1 epoch on the 122k-doc mixed corpus (coai / storyscope / gutenberg / blogs / scp).
fp16 on T4; NaN preflight aborts loudly rather than silently producing garbage.

Exports `artifacts/lfm/` onto Google Drive `MyDrive/isthisaislop/`.


In [ ]:
# Config
EPOCHS = 1
MAX_LEN = 512
MODEL = "LiquidAI/LFM2.5-Encoder-350M"
DATA_PARQUET = "train_all.parquet"   # name of the training parquet (Drive or upload)
DRIVE_PATH = "isthisaislop"          # folder under MyDrive
MOUNT_DRIVE = True

import os, sys
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    drive = None

if IN_COLAB and MOUNT_DRIVE:
    try:
        drive.mount("/content/drive")
        ROOT = Path("/content/drive/MyDrive") / DRIVE_PATH
    except Exception as exc:
        print("Drive mount failed, using /content:", exc)
        ROOT = Path("/content/isthisaislop")
elif IN_COLAB:
    ROOT = Path("/content/isthisaislop")
else:
    ROOT = Path(".").resolve()
    if not (ROOT / "src" / "slopdet").exists():
        ROOT = Path("/home/vstaln/slop-detector")

ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)
print("ROOT", ROOT)

import torch
if torch.cuda.is_available():
    print("GPU", torch.cuda.get_device_name(0))
else:
    raise SystemExit("No GPU. Runtime → Change runtime type → T4 GPU → Save → Run all.")

import subprocess
pkgs = ["pyyaml", "regex", "jsonschema", "scikit-learn", "numpy>=1.26", "pandas", "pyarrow"]
pkgs += ["transformers", "accelerate", "safetensors"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
print("deps ok")


In [ ]:
# Write the package onto disk (self-contained; no git clone required)
import json
from pathlib import Path
FILES = json.loads('{"src/slopdet/__init__.py": "\\"\\"\\"ITAIS (Is This AI Slop?) \\u2014 classifier + checkable why. Import: slopdet.\\"\\"\\"\\n\\n__version__ = \\"0.1.0\\"\\n", "src/slopdet/calibrate.py": "\\"\\"\\"Calibration helpers for matches_ai_pile (1% FPR on human reference).\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom typing import Any\\n\\n\\ndef threshold_at_fpr(human_scores: list[float], fpr: float = 0.01) -> float:\\n    if not human_scores:\\n        return 1.0\\n    ordered = sorted(human_scores, reverse=True)\\n    k = max(0, min(len(ordered) - 1, int(len(ordered) * fpr)))\\n    return float(ordered[k])\\n\\n\\ndef human_percentile(score: float, human_scores: list[float]) -> float:\\n    if not human_scores:\\n        return 0.0\\n    n = sum(1 for s in human_scores if score > s)\\n    return 100.0 * n / len(human_scores)\\n\\n\\ndef calibration_record(human_scores: list[float], fpr: float = 0.01) -> dict[str, Any]:\\n    return {\\n        \\"fpr\\": fpr,\\n        \\"threshold\\": threshold_at_fpr(human_scores, fpr),\\n        \\"n_human\\": len(human_scores),\\n        \\"label\\": \\"matches_ai_pile\\",\\n    }\\n", "src/slopdet/span.py": "\\"\\"\\"Stitch human and AI sentences so a token classifier can mark which spans.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport random\\nimport re\\nfrom typing import Any\\n\\n_SENT_SPLIT = re.compile(r\\"(?<=[.!?])\\\\s+\\")\\n\\n\\ndef split_sentences(text: str) -> list[str]:\\n    parts = [p.strip() for p in _SENT_SPLIT.split(text.strip()) if p.strip()]\\n    return parts or ([text.strip()] if text.strip() else [])\\n\\n\\ndef stitch_docs(\\n    docs: list[dict[str, Any]],\\n    rng: random.Random,\\n    n_human: int = 3,\\n    n_ai: int = 3,\\n    limit: int | None = None,\\n) -> list[dict[str, Any]]:\\n    \\"\\"\\"Splice human + AI sentences into mixed docs with per-sentence labels (0=human, 1=ai).\\"\\"\\"\\n    human = [d[\\"text\\"] for d in docs if int(d[\\"label\\"]) == 0]\\n    ai = [d[\\"text\\"] for d in docs if int(d[\\"label\\"]) == 1]\\n    rng.shuffle(human)\\n    rng.shuffle(ai)\\n    mixed: list[dict[str, Any]] = []\\n    n_pairs = min(len(human), len(ai))\\n    if limit is not None:\\n        n_pairs = min(n_pairs, limit)\\n    for i in range(n_pairs):\\n        h_sents = split_sentences(human[i])\\n        a_sents = split_sentences(ai[i])\\n        if len(h_sents) < n_human or len(a_sents) < n_ai:\\n            continue\\n        sentences = [(s, 0) for s in h_sents[:n_human]] + [(s, 1) for s in a_sents[:n_ai]]\\n        rng.shuffle(sentences)\\n        text, offset, spans = \\"\\", 0, []\\n        for sent, lab in sentences:\\n            spans.append((offset, offset + len(sent), lab))\\n            text += sent + \\" \\"\\n            offset += len(sent) + 1\\n        mixed.append(\\n            {\\n                \\"id\\": f\\"stitch-{i}\\",\\n                \\"text\\": text.strip(),\\n                \\"sentences\\": spans,\\n                \\"source\\": \\"stitched\\",\\n            }\\n        )\\n    return mixed\\n\\n\\ndef pure_docs(docs: list[dict[str, Any]]) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:\\n    \\"\\"\\"Whole-document spans labeled with the source pile. Used for calibration, not training.\\"\\"\\"\\n    human: list[dict[str, Any]] = []\\n    ai: list[dict[str, Any]] = []\\n    for i, doc in enumerate(docs):\\n        text = str(doc[\\"text\\"])\\n        lab = int(doc[\\"label\\"])\\n        rec = {\\n            \\"id\\": doc.get(\\"id\\", f\\"pure-{lab}-{i}\\"),\\n            \\"text\\": text,\\n            \\"sentences\\": [(0, len(text), lab)],\\n            \\"source\\": \\"pure\\",\\n            \\"label\\": lab,\\n        }\\n        (human if lab == 0 else ai).append(rec)\\n    return human, ai\\n\\n\\ndef token_labels(doc: dict[str, Any], tokenizer: Any, max_len: int) -> dict[str, list]:\\n    \\"\\"\\"Map per-sentence source labels to per-token labels. -100 = special/padding (ignored).\\"\\"\\"\\n    enc = tokenizer(doc[\\"text\\"], truncation=True, max_length=max_len, return_offsets_mapping=True)\\n    input_ids, attn = enc[\\"input_ids\\"], enc[\\"attention_mask\\"]\\n    labels = [-100] * len(input_ids)\\n    offsets = enc[\\"offset_mapping\\"]\\n    for i, (start, end) in enumerate(offsets):\\n        if i == 0 or i == len(offsets) - 1 or start == end:\\n            continue\\n        for s, e, lab in doc[\\"sentences\\"]:\\n            if s <= start < e:\\n                labels[i] = lab\\n                break\\n    return {\\"input_ids\\": input_ids, \\"attention_mask\\": attn, \\"labels\\": labels}\\n", "src/slopdet/construction.py": "\\"\\"\\"Cheap deterministic construction stats. No spaCy, no GPU.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport math\\nimport re\\nfrom typing import Any\\n\\n_SENT_SPLIT = re.compile(r\\"(?<=[.!?])\\\\s+\\")\\n_PARA_SPLIT = re.compile(r\\"\\\\n\\\\s*\\\\n\\")\\n_WORD = re.compile(r\\"[A-Za-z\']+\\")\\n_PROPER = re.compile(r\\"\\\\b[A-Z][a-z]{2,}\\\\b\\")\\n_DIGIT = re.compile(r\\"\\\\d\\")\\n_DATE = re.compile(\\n    r\\"\\\\b(?:(?:jan|feb|mar|apr|may|jun|jul|aug|sep|sept|oct|nov|dec)[a-z]*\\\\.?\\\\s+\\\\d{1,2}\\"\\n    r\\"|\\\\d{1,2}/\\\\d{1,2}/\\\\d{2,4}|\\\\d{4})\\\\b\\",\\n    re.I,\\n)\\n_SUBORD = re.compile(\\n    r\\"\\\\b(?:because|although|though|unless|while|whereas|if|when|after|before|since)\\\\b\\",\\n    re.I,\\n)\\n_CLOSURE = re.compile(\\n    r\\"\\\\b(?:in conclusion|ultimately|overall|to sum up|in summary|to conclude)\\\\b\\",\\n    re.I,\\n)\\n_OVER_EXPLAIN = re.compile(\\n    r\\"\\\\b(?:the key point is|as you can see|this distinction matters|in other words|\\"\\n    r\\"highlighting|underscoring|reflecting|showcasing)\\\\b\\",\\n    re.I,\\n)\\n\\n\\ndef _sentences(text: str) -> list[str]:\\n    parts = [s.strip() for s in _SENT_SPLIT.split(text.strip()) if s.strip()]\\n    return parts or ([text.strip()] if text.strip() else [])\\n\\n\\ndef _paragraphs(text: str) -> list[str]:\\n    parts = [p.strip() for p in _PARA_SPLIT.split(text) if p.strip()]\\n    return parts or ([text.strip()] if text.strip() else [])\\n\\n\\ndef _cosine(a: list[float], b: list[float]) -> float:\\n    dot = sum(x * y for x, y in zip(a, b))\\n    na = math.sqrt(sum(x * x for x in a))\\n    nb = math.sqrt(sum(y * y for y in b))\\n    if na == 0 or nb == 0:\\n        return 0.0\\n    return dot / (na * nb)\\n\\n\\ndef _para_features(para: str) -> list[float]:\\n    sents = _sentences(para)\\n    lengths = [len(s.split()) for s in sents] or [0]\\n    words = _WORD.findall(para.lower())\\n    n = max(len(words), 1)\\n    mean_len = sum(lengths) / max(len(lengths), 1)\\n    ttr = len(set(words)) / n\\n    comma = para.count(\\",\\") / n\\n    sub = len(_SUBORD.findall(para)) / n\\n    mean_word = sum(len(w) for w in words) / n\\n    return [mean_len, ttr, comma, sub, mean_word]\\n\\n\\ndef over_explain_spans(text: str) -> list[dict[str, Any]]:\\n    \\"\\"\\"Verbatim spans of over-explain phrases, e.g. \'the key point is\'.\\"\\"\\"\\n    return [\\n        {\\"start\\": m.start(), \\"end\\": m.end(), \\"quote\\": text[m.start() : m.end()]}\\n        for m in _OVER_EXPLAIN.finditer(text)\\n    ]\\n\\n\\ndef construction_stats(text: str) -> dict[str, Any]:\\n    sents = _sentences(text)\\n    lengths = [len(s.split()) for s in sents]\\n    mean = (sum(lengths) / len(lengths)) if lengths else 0.0\\n    if len(lengths) >= 2:\\n        var = sum((x - mean) ** 2 for x in lengths) / len(lengths)\\n        std = math.sqrt(var)\\n        burstiness = std / mean if mean else 0.0\\n        adjacent_contrast = sum(\\n            1 for a, b in zip(lengths, lengths[1:]) if abs(a - b) >= 20\\n        )\\n    else:\\n        burstiness = 0.0\\n        adjacent_contrast = 0\\n\\n    paras = _paragraphs(text)\\n    vecs = [_para_features(p) for p in paras]\\n    if len(vecs) >= 2:\\n        sims = [\\n            _cosine(vecs[i], vecs[j])\\n            for i in range(len(vecs))\\n            for j in range(i + 1, len(vecs))\\n        ]\\n        evenness = sum(sims) / len(sims)\\n    else:\\n        evenness = 0.0\\n\\n    if paras:\\n        last = paras[-1]\\n        earlier_openers = \\" \\".join(_sentences(p)[:1][0] if _sentences(p) else \\"\\" for p in paras[:-1])\\n        last_words = set(_WORD.findall(last.lower()))\\n        earlier_words = set(_WORD.findall(earlier_openers.lower()))\\n        overlap = len(last_words & earlier_words) / max(len(last_words), 1)\\n        recap_closure = overlap + (0.5 if _CLOSURE.search(last) else 0.0)\\n    else:\\n        recap_closure = 0.0\\n\\n    n_words = max(len(text.split()), 1)\\n    over_explain = 1000.0 * len(_OVER_EXPLAIN.findall(text)) / n_words\\n\\n    portable = 0\\n    for sent in sents:\\n        if not (_PROPER.search(sent) or _DIGIT.search(sent) or _DATE.search(sent)):\\n            portable += 1\\n    portability = portable / max(len(sents), 1)\\n\\n    return {\\n        \\"burstiness\\": round(burstiness, 6),\\n        \\"adjacent_contrast\\": int(adjacent_contrast),\\n        \\"evenness\\": round(evenness, 6),\\n        \\"recap_closure\\": round(recap_closure, 6),\\n        \\"over_explain\\": round(over_explain, 6),\\n        \\"portability\\": round(portability, 6),\\n        \\"n_sentences\\": len(sents),\\n        \\"n_paragraphs\\": len(paras),\\n        \\"n_words\\": n_words,\\n    }\\n", "src/slopdet/ontology.py": "\\"\\"\\"Load and freeze the slop pattern ontology.\\n\\nIds are append-only. Disable a pattern with enabled=false; do not delete it.\\nONTOLOGY_SHA256 is sha256 of the three YAML files concatenated in this order:\\npatterns.core.yaml, patterns.wikipedia.yaml, patterns.rhetorical.yaml.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport json\\nfrom dataclasses import dataclass, field\\nfrom functools import lru_cache\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport regex as regex_mod\\nimport yaml\\nfrom jsonschema import Draft202012Validator\\n\\nYAML_NAMES = (\\n    \\"patterns.core.yaml\\",\\n    \\"patterns.wikipedia.yaml\\",\\n    \\"patterns.rhetorical.yaml\\",\\n    \\"patterns.slop.yaml\\",\\n)\\n\\n\\ndef default_ontology_dir() -> Path:\\n    return Path(__file__).resolve().parents[2] / \\"ontology\\"\\n\\n\\nclass OntologyError(ValueError):\\n    \\"\\"\\"Invalid ontology data.\\"\\"\\"\\n\\n\\n@dataclass(frozen=True)\\nclass Pattern:\\n    id: str\\n    lane: str\\n    unit: str\\n    detector: str\\n    pattern: str\\n    fix: str\\n    source: str\\n    license: str\\n    min_len_words: int\\n    paper: str | None\\n    enabled: bool\\n    compiled: Any = field(repr=False, compare=False)\\n\\n\\n@dataclass(frozen=True)\\nclass Ontology:\\n    patterns: tuple[Pattern, ...]\\n    sha256: str\\n    by_id: dict[str, Pattern]\\n\\n    @property\\n    def ONTOLOGY_SHA256(self) -> str:\\n        return self.sha256\\n\\n    def enabled_patterns(self) -> tuple[Pattern, ...]:\\n        return tuple(p for p in self.patterns if p.enabled)\\n\\n\\ndef _schema(ontology_dir: Path) -> dict[str, Any]:\\n    return json.loads((ontology_dir / \\"schema.json\\").read_text(encoding=\\"utf-8\\"))\\n\\n\\ndef _load_yaml_list(path: Path) -> list[dict[str, Any]]:\\n    data = yaml.safe_load(path.read_text(encoding=\\"utf-8\\"))\\n    if not isinstance(data, list):\\n        raise OntologyError(f\\"{path.name} must be a YAML list, got {type(data).__name__}\\")\\n    return data\\n\\n\\n@lru_cache(maxsize=4)\\ndef load_ontology(ontology_dir: Path | None = None) -> Ontology:\\n    ontology_dir = Path(ontology_dir) if ontology_dir else default_ontology_dir()\\n    validator = Draft202012Validator(_schema(ontology_dir))\\n    seen: dict[str, str] = {}\\n    patterns: list[Pattern] = []\\n    raw_parts: list[bytes] = []\\n\\n    for name in YAML_NAMES:\\n        path = ontology_dir / name\\n        raw_parts.append(path.read_bytes())\\n        for i, entry in enumerate(_load_yaml_list(path)):\\n            errors = sorted(validator.iter_errors(entry), key=lambda e: list(e.path))\\n            if errors:\\n                raise OntologyError(f\\"{name}[{i}]: {errors[0].message}\\")\\n            pid = entry[\\"id\\"]\\n            if pid in seen:\\n                raise OntologyError(f\\"duplicate id {pid!r} in {name} and {seen[pid]}\\")\\n            seen[pid] = name\\n            try:\\n                compiled = regex_mod.compile(entry[\\"pattern\\"])\\n            except regex_mod.error as exc:\\n                raise OntologyError(f\\"{pid}: regex does not compile: {exc}\\") from exc\\n            patterns.append(\\n                Pattern(\\n                    id=pid,\\n                    lane=entry[\\"lane\\"],\\n                    unit=entry[\\"unit\\"],\\n                    detector=entry[\\"detector\\"],\\n                    pattern=entry[\\"pattern\\"],\\n                    fix=entry[\\"fix\\"],\\n                    source=entry[\\"source\\"],\\n                    license=entry[\\"license\\"],\\n                    min_len_words=int(entry[\\"min_len_words\\"]),\\n                    paper=entry[\\"paper\\"],\\n                    enabled=bool(entry[\\"enabled\\"]),\\n                    compiled=compiled,\\n                )\\n            )\\n\\n    sha256 = hashlib.sha256(b\\"\\".join(raw_parts)).hexdigest()\\n    frozen = tuple(patterns)\\n    return Ontology(\\n        patterns=frozen,\\n        sha256=sha256,\\n        by_id={p.id: p for p in frozen},\\n    )\\n\\n\\nONTOLOGY_SHA256: str | None = None\\n\\n\\ndef ontology_sha256(ontology_dir: Path | None = None) -> str:\\n    global ONTOLOGY_SHA256\\n    if ONTOLOGY_SHA256 is None:\\n        ONTOLOGY_SHA256 = load_ontology(ontology_dir).sha256\\n    return ONTOLOGY_SHA256\\n", "src/slopdet/weaklabel.py": "\\"\\"\\"Span weak-labeler. Overlapping hits from different ids all survive.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom typing import Any\\n\\nfrom slopdet.ontology import Ontology, Pattern\\n\\n\\ndef _word_count(text: str) -> int:\\n    return len(text.split())\\n\\n\\ndef label_text(text: str, ontology: Ontology, *, enabled_only: bool = True) -> list[dict[str, Any]]:\\n    n_words = _word_count(text)\\n    patterns: tuple[Pattern, ...] = (\\n        ontology.enabled_patterns() if enabled_only else ontology.patterns\\n    )\\n    hits: list[dict[str, Any]] = []\\n    for pattern in patterns:\\n        if n_words < pattern.min_len_words:\\n            continue\\n        for match in pattern.compiled.finditer(text):\\n            start, end = match.start(), match.end()\\n            if end <= start:\\n                continue\\n            hits.append(\\n                {\\n                    \\"id\\": pattern.id,\\n                    \\"start\\": start,\\n                    \\"end\\": end,\\n                    \\"unit\\": pattern.unit,\\n                    \\"lane\\": pattern.lane,\\n                    \\"quote\\": text[start:end],\\n                    \\"fix\\": pattern.fix,\\n                }\\n            )\\n    hits.sort(key=lambda h: (h[\\"start\\"], h[\\"end\\"], h[\\"id\\"]))\\n    return hits\\n", "src/slopdet/report.py": "\\"\\"\\"Hit renderer. Two lanes never merge. Never emit a percentage-of-AI string.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom typing import Any\\n\\nFORBIDDEN_SUBSTRINGS = (\\n    \\"% AI\\",\\n    \\"% ai\\",\\n    \\"percent AI\\",\\n    \\"AI-generated\\",\\n    \\"ai-generated\\",\\n    \\"written by ChatGPT\\",\\n    \\"written by GPT\\",\\n)\\n\\n\\ndef render_hits(\\n    hits: list[dict[str, Any]],\\n    *,\\n    resemblance: dict[str, Any] | None = None,\\n) -> dict[str, Any]:\\n    style = [h for h in hits if h.get(\\"lane\\") != \\"resemblance\\"]\\n    out: dict[str, Any] = {\\n        \\"status\\": \\"ok\\",\\n        \\"hits\\": [\\n            {\\n                \\"id\\": h[\\"id\\"],\\n                \\"lane\\": h.get(\\"lane\\", \\"style\\"),\\n                \\"unit\\": h.get(\\"unit\\", \\"span\\"),\\n                \\"quote\\": h.get(\\"quote\\", \\"\\"),\\n                \\"say\\": h.get(\\"say\\") or h.get(\\"fix\\", \\"\\"),\\n                \\"fix\\": h.get(\\"fix\\") or h.get(\\"say\\", \\"\\"),\\n            }\\n            for h in style\\n        ],\\n        \\"style_summary\\": \\"Nothing matched.\\" if not style else None,\\n        \\"resemblance\\": None,\\n    }\\n    if not style:\\n        out[\\"style_summary\\"] = \\"Nothing matched.\\"\\n    if resemblance is not None:\\n        pct = resemblance.get(\\"human_percentile\\")\\n        if pct is None:\\n            out[\\"resemblance\\"] = {\\n                \\"label\\": \\"matches_ai_pile\\",\\n                \\"text\\": resemblance.get(\\"text\\", \\"Resemblance unavailable.\\"),\\n            }\\n        else:\\n            out[\\"resemblance\\"] = {\\n                \\"label\\": \\"matches_ai_pile\\",\\n                \\"text\\": (\\n                    f\\"Resembles the AI pile more than {pct:.0f}% of human reference texts.\\"\\n                ),\\n            }\\n    # The guard applies to OUR copy (say/fix/summaries/resemblance/labels), never\\n    # to verbatim user quotes: a span that literally quotes \\"AI-generated\\" from\\n    # the user\'s own text is not a claim we make, and crashing the labeling\\n    # pipeline on it would be a false positive (StoryScope fiction trips this).\\n    # Scanned from the generated fields directly (not `str(dict)`, whose repr\\n    # escaping breaks substring matching on multi-line quotes).\\n    bits: list[str] = [str(out.get(\\"status\\", \\"\\")), str(out.get(\\"style_summary\\") or \\"\\")]\\n    for hit in out[\\"hits\\"]:\\n        for key in (\\"id\\", \\"lane\\", \\"unit\\", \\"say\\", \\"fix\\"):\\n            bits.append(str(hit.get(key, \\"\\")))\\n    res = out.get(\\"resemblance\\") or {}\\n    bits += [str(res.get(\\"label\\", \\"\\")), str(res.get(\\"text\\", \\"\\"))]\\n    blob = \\"\\\\n\\".join(bits)\\n    for bad in FORBIDDEN_SUBSTRINGS:\\n        if bad in blob and \\"AI pile\\" not in bad:\\n            holder = next(\\n                (k for k, v in out.items() if bad in str(v)),\\n                \\"hits[].say/fix\\",\\n            )\\n            raise ValueError(f\\"forbidden copy leaked: {bad!r} (in {holder})\\")\\n    return out\\n", "src/slopdet/scorer.py": "\\"\\"\\"CPU pile-resemblance scorer. Numpy at inference; sklearn only to train.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nimport math\\nfrom functools import lru_cache\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nfrom slopdet.calibrate import human_percentile\\nfrom slopdet.construction import construction_stats\\nfrom slopdet.ontology import Ontology, load_ontology\\nfrom slopdet.weaklabel import label_text\\n\\nCONSTRUCTION_KEYS = (\\n    \\"burstiness\\",\\n    \\"evenness\\",\\n    \\"recap_closure\\",\\n    \\"over_explain\\",\\n    \\"portability\\",\\n)\\nN_EXTRA = len(CONSTRUCTION_KEYS) + 1  # + n_words/1000\\n\\n\\ndef default_bundle_path() -> Path:\\n    return Path(__file__).resolve().parents[2] / \\"artifacts\\" / \\"sklearn_bundle.json\\"\\n\\n\\ndef featurize(\\n    text: str,\\n    ontology: Ontology,\\n    pattern_ids: list[str],\\n) -> list[float]:\\n    hits = label_text(text, ontology)\\n    counts = dict.fromkeys(pattern_ids, 0.0)\\n    for hit in hits:\\n        if hit[\\"id\\"] in counts:\\n            counts[hit[\\"id\\"]] += 1.0\\n    stats = construction_stats(text)\\n    extra = [float(stats.get(k) or 0.0) for k in CONSTRUCTION_KEYS]\\n    extra.append(float(stats.get(\\"n_words\\") or 0.0) / 1000.0)\\n    return [counts[pid] for pid in pattern_ids] + extra\\n\\n\\ndef _sigmoid(x: float) -> float:\\n    if x >= 0:\\n        z = math.exp(-x)\\n        return 1.0 / (1.0 + z)\\n    z = math.exp(x)\\n    return z / (1.0 + z)\\n\\n\\n@lru_cache(maxsize=1)\\ndef load_bundle(path: Path | None = None) -> dict[str, Any]:\\n    path = path or default_bundle_path()\\n    return json.loads(path.read_text(encoding=\\"utf-8\\"))\\n\\n\\ndef score_vector(vec: list[float], bundle: dict[str, Any]) -> float:\\n    mean = bundle[\\"scaler_mean\\"]\\n    scale = bundle[\\"scaler_scale\\"]\\n    coef = bundle[\\"coef\\"]\\n    if len(vec) != len(coef):\\n        raise ValueError(f\\"feature dim {len(vec)} != coef dim {len(coef)}\\")\\n    acc = float(bundle[\\"intercept\\"])\\n    for v, m, s, c in zip(vec, mean, scale, coef):\\n        acc += c * ((v - m) / (s if s else 1.0))\\n    return _sigmoid(acc)\\n\\n\\ndef score_text(\\n    text: str,\\n    *,\\n    ontology: Ontology | None = None,\\n    bundle: dict[str, Any] | None = None,\\n    bundle_path: Path | None = None,\\n) -> dict[str, Any]:\\n    bundle = bundle or load_bundle(bundle_path)\\n    onto = ontology or load_ontology()\\n    vec = featurize(text, onto, bundle[\\"pattern_ids\\"])\\n    score = score_vector(vec, bundle)\\n    human_scores = bundle.get(\\"calibration\\", {}).get(\\"human_scores\\") or []\\n    pct = human_percentile(score, human_scores)\\n    return {\\n        \\"label\\": \\"matches_ai_pile\\",\\n        \\"score\\": score,\\n        \\"human_percentile\\": pct,\\n        \\"text\\": (\\n            f\\"Resembles the AI pile more than {pct:.0f}% of human reference texts.\\"\\n        ),\\n        \\"trained_on\\": bundle.get(\\"trained_on\\"),\\n    }\\n", "src/slopdet/explain.py": "\\"\\"\\"Checkable why-slop and why-human. Never an authorship claim.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nimport re\\nfrom typing import Any\\n\\nfrom slopdet.construction import construction_stats, over_explain_spans\\nfrom slopdet.ontology import Ontology, default_ontology_dir, load_ontology\\nfrom slopdet.report import render_hits\\nfrom slopdet.scorer import default_bundle_path, load_bundle, score_text\\nfrom slopdet.span import split_sentences\\nfrom slopdet.storyscope import storyscope_hits\\nfrom slopdet.tags import COPY, pack_style, say\\nfrom slopdet.weaklabel import label_text\\n\\n# Density proxies that fire on almost all academic prose. Checkable, but not slop evidence.\\nREGISTER_IDS = frozenset(\\n    {\\n        \\"rhet_nominalization_density\\",\\n        \\"rhet_copula_avoidance\\",\\n    }\\n)\\n\\n_WEEKDAY = re.compile(\\n    r\\"\\\\b(?:Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday)s?\\\\b\\"\\n)\\n_CLOCK = re.compile(r\\"\\\\b(?:\\\\d{1,2}(?::\\\\d{2})?\\\\s*(?:am|pm)|3am|noon|midnight)\\\\b\\", re.I)\\n_DAYPART = re.compile(r\\"\\\\b(?:mornings?|afternoons?|evenings?|nights?)\\\\b\\", re.I)\\n_CONTRACTION = re.compile(\\n    r\\"\\\\b(?:don\'t|doesn\'t|didn\'t|I\'m|I\'ve|I\'d|wasn\'t|aren\'t|isn\'t|won\'t|can\'t)\\\\b\\",\\n    re.I,\\n)\\n_FIRST_PERSON = re.compile(r\\"\\\\bI\\\\b\\")\\n_PROPER_INNER = re.compile(\\n    r\\"(?<!\\\\A)(?<![.!?]\\\\s)(?<![\\\\\\"\\\\u201c\\\\u201d(\\\\[])\\\\b[A-Z][a-z]{2,}\\\\b\\"\\n)\\n_DIGIT = re.compile(r\\"\\\\b\\\\d+(?:[./:]\\\\d+)*\\\\b\\")\\n\\n\\ndef _slop_hits(hits: list[dict[str, Any]]) -> list[dict[str, Any]]:\\n    return [h for h in hits if h[\\"id\\"] not in REGISTER_IDS]\\n\\n\\ndef _human_signals(text: str, stats: dict[str, Any]) -> list[dict[str, Any]]:\\n    found: list[dict[str, Any]] = []\\n    seen: set[tuple[str, int, int]] = set()\\n\\n    def add(pid: str, start: int, end: int) -> None:\\n        key = (pid, start, end)\\n        if key in seen or end <= start:\\n            return\\n        seen.add(key)\\n        spec = COPY[pid]\\n        found.append(\\n            {\\n                \\"id\\": pid,\\n                \\"lane\\": spec[\\"lane\\"],\\n                \\"unit\\": \\"span\\",\\n                \\"quote\\": text[start:end],\\n                \\"lean\\": spec[\\"lean\\"],\\n                \\"say\\": spec[\\"say\\"],\\n            }\\n        )\\n\\n    for rx, pid in (\\n        (_WEEKDAY, \\"weekday\\"),\\n        (_CLOCK, \\"clock\\"),\\n        (_DAYPART, \\"daypart\\"),\\n        (_CONTRACTION, \\"spoken\\"),\\n        (_DIGIT, \\"number\\"),\\n    ):\\n        for match in rx.finditer(text):\\n            add(pid, match.start(), match.end())\\n\\n    name_hits = 0\\n    for match in _PROPER_INNER.finditer(text):\\n        # Cap at one: repeated title-case terms are register (Mixed-Integer\\n        # Programming), not names. A single mid-sentence capital is a real cue.\\n        if name_hits >= 1:\\n            break\\n        add(\\"name\\", match.start(), match.end())\\n        name_hits += 1\\n\\n    if _FIRST_PERSON.search(text) and stats.get(\\"n_words\\", 0) <= 80:\\n        match = _FIRST_PERSON.search(text)\\n        if match:\\n            add(\\"first\\", match.start(), match.end())\\n\\n    if stats.get(\\"n_sentences\\", 0) >= 3 and float(stats.get(\\"burstiness\\") or 0) >= 0.25:\\n        found.append(\\n            {\\n                \\"id\\": \\"burst\\",\\n                \\"lane\\": \\"construction\\",\\n                \\"lean\\": \\"human\\",\\n                \\"unit\\": \\"piece\\",\\n                \\"quote\\": \\"\\",\\n                \\"say\\": say(\\"burst\\"),\\n            }\\n        )\\n    if int(stats.get(\\"adjacent_contrast\\") or 0) >= 1:\\n        found.append(\\n            {\\n                \\"id\\": \\"contrast\\",\\n                \\"lane\\": \\"construction\\",\\n                \\"lean\\": \\"human\\",\\n                \\"unit\\": \\"piece\\",\\n                \\"quote\\": \\"\\",\\n                \\"say\\": say(\\"contrast\\"),\\n            }\\n        )\\n    return found\\n\\n\\ndef _construction_slop(text: str, stats: dict[str, Any]) -> list[dict[str, Any]]:\\n    out: list[dict[str, Any]] = []\\n    if float(stats.get(\\"recap_closure\\") or 0) >= 0.5:\\n        out.append(\\n            {\\n                \\"id\\": \\"recap\\",\\n                \\"lane\\": \\"construction\\",\\n                \\"lean\\": \\"slop\\",\\n                \\"unit\\": \\"paragraph\\",\\n                \\"quote\\": text[-min(80, len(text)) :],\\n                \\"say\\": say(\\"recap\\"),\\n                \\"fix\\": say(\\"recap\\"),\\n            }\\n        )\\n    if float(stats.get(\\"over_explain\\") or 0) > 0:\\n        spans = over_explain_spans(text)\\n        quote = spans[0][\\"quote\\"] if spans else \\"\\"\\n        out.append(\\n            {\\n                \\"id\\": \\"gloss\\",\\n                \\"lane\\": \\"construction\\",\\n                \\"lean\\": \\"slop\\",\\n                \\"unit\\": \\"span\\",\\n                \\"start\\": spans[0][\\"start\\"] if spans else None,\\n                \\"end\\": spans[0][\\"end\\"] if spans else None,\\n                \\"quote\\": quote,\\n                \\"say\\": say(\\"gloss\\"),\\n                \\"fix\\": say(\\"gloss\\"),\\n            }\\n        )\\n    if (\\n        int(stats.get(\\"n_sentences\\") or 0) >= 4\\n        and float(stats.get(\\"burstiness\\") or 0) < 0.12\\n    ):\\n        out.append(\\n            {\\n                \\"id\\": \\"even\\",\\n                \\"lane\\": \\"construction\\",\\n                \\"lean\\": \\"slop\\",\\n                \\"unit\\": \\"piece\\",\\n                \\"quote\\": \\"\\",\\n                \\"say\\": say(\\"even\\"),\\n                \\"fix\\": say(\\"even\\"),\\n            }\\n        )\\n    return out\\n\\n\\ndef _lean(why_slop: list[dict[str, Any]], why_human: list[dict[str, Any]]) -> str:\\n    if why_slop and why_human:\\n        return \\"mixed\\"\\n    if why_slop:\\n        return \\"slop\\"\\n    if why_human:\\n        return \\"human\\"\\n    return \\"unclear\\"\\n\\n\\ndef _doc_lean(sentences: list[dict[str, Any]]) -> str:\\n    leans = {s[\\"lean\\"] for s in sentences}\\n    if \\"slop\\" in leans and \\"human\\" in leans:\\n        return \\"mixed\\"\\n    if \\"slop\\" in leans:\\n        return \\"slop\\"\\n    if \\"human\\" in leans:\\n        return \\"human\\"\\n    return \\"unclear\\"\\n\\n\\ndef _pile_resemblance(text: str, onto: Ontology) -> dict[str, Any] | None:\\n    path = default_bundle_path()\\n    if not path.is_file():\\n        return None\\n    try:\\n        bundle = load_bundle(path)\\n    except (OSError, json.JSONDecodeError, KeyError, ValueError):\\n        return None\\n    if bundle.get(\\"ontology_sha256\\") != onto.sha256:\\n        return None\\n    scored = score_text(text, ontology=onto, bundle=bundle)\\n    return {\\n        \\"label\\": scored[\\"label\\"],\\n        \\"text\\": scored[\\"text\\"],\\n        \\"human_percentile\\": scored[\\"human_percentile\\"],\\n        \\"trained_on\\": scored.get(\\"trained_on\\"),\\n    }\\n\\n\\ndef _sentence_record(sentence: str, ontology: Ontology) -> dict[str, Any]:\\n    hits = _slop_hits(label_text(sentence, ontology))\\n    stats = construction_stats(sentence)\\n    why_human = _human_signals(sentence, stats)\\n    why_slop = [pack_style(h) for h in hits]\\n    human_leaned = [h for h in why_slop if h.get(\\"lean\\") == \\"human\\"]\\n    why_slop = [h for h in why_slop if h.get(\\"lean\\") != \\"human\\"]\\n    why_human.extend(human_leaned)\\n    return {\\n        \\"text\\": sentence,\\n        \\"lean\\": _lean(why_slop, why_human),\\n        \\"why_slop\\": why_slop,\\n        \\"why_human\\": why_human,\\n    }\\n\\n\\ndef explain(\\n    text: str, ontology_dir: Any | None = None, *, sentences: bool = True\\n) -> dict[str, Any]:\\n    onto = load_ontology(ontology_dir or default_ontology_dir())\\n    raw_hits = label_text(text, onto)\\n    slop_hits = _slop_hits(raw_hits)\\n    stats = construction_stats(text)\\n    why_slop = [pack_style(h) for h in slop_hits]\\n    # A pattern\'s lean is authoritative: human-lean hits (weasel, frames,\\n    # passive) vote human, not slop. Partition here so _lean() and the\\n    # slop/human tag columns agree with the per-span lean.\\n    human_leaned = [h for h in why_slop if h.get(\\"lean\\") == \\"human\\"]\\n    why_slop = [h for h in why_slop if h.get(\\"lean\\") != \\"human\\"]\\n    why_slop.extend(_construction_slop(text, stats))\\n    why_human = _human_signals(text, stats)\\n    why_human.extend(human_leaned)\\n    for hit in storyscope_hits(text):\\n        spec = COPY[hit[\\"id\\"]]\\n        hit.setdefault(\\"say\\", spec[\\"say\\"])\\n        hit.setdefault(\\"fix\\", spec[\\"say\\"])\\n        (why_slop if hit[\\"lean\\"] == \\"slop\\" else why_human).append(hit)\\n    if sentences:\\n        sentence_records = [_sentence_record(s, onto) for s in split_sentences(text)]\\n        lean = _doc_lean(sentence_records)\\n    else:\\n        sentence_records = []\\n        lean = _lean(why_slop, why_human)\\n    rendered = render_hits(why_slop, resemblance=_pile_resemblance(text, onto))\\n    rendered.update(\\n        {\\n            \\"lean\\": lean,\\n            \\"why_slop\\": why_slop,\\n            \\"why_human\\": why_human,\\n            \\"sentences\\": sentence_records,\\n            \\"construction\\": stats,\\n            \\"ontology_sha256\\": onto.sha256,\\n        }\\n    )\\n    return rendered\\n", "src/slopdet/labels.py": "\\"\\"\\"Shared label parsing for training data.\\n\\nK1 fix: `scripts/fine_tune_lfm.py` and `scripts/build_training_parquet.py` used to\\nparse the same `label`/`pile` columns with opposite missing-value defaults\\n(missing -> AI in one, missing -> human in the other). A malformed or renamed\\ncolumn could silently flip thousands of labels between the two entry points.\\n\\nBoth scripts now call `parse_label` so a doc has exactly one interpretation\\neverywhere: explicit `label` wins; else `pile` with a strict enum; else the\\ncaller\'s explicit default. Anything unrecognized raises.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\n# pile values -> label (0 = human, 1 = AI). Accepts ints, floats, bools, strings.\\n_PILE_TO_LABEL: dict[str, int] = {\\"0\\": 0, \\"1\\": 1, \\"0.0\\": 0, \\"1.0\\": 1,\\n                                  \\"human\\": 0, \\"ai\\": 1, \\"false\\": 0, \\"true\\": 1}\\n\\n\\ndef parse_label(rec: dict, default: int) -> int:\\n    \\"\\"\\"Return the 0/1 label for a training row, or raise on an unrecognized value.\\n\\n    Precedence: explicit ``label`` column > ``pile`` column > ``default``.\\n    \\"\\"\\"\\n    if \\"label\\" in rec and rec[\\"label\\"] is not None:\\n        try:\\n            return int(rec[\\"label\\"])\\n        except (TypeError, ValueError) as exc:\\n            raise ValueError(f\\"unrecognized label {rec[\'label\']!r}\\") from exc\\n    if \\"pile\\" in rec and rec[\\"pile\\"] is not None:\\n        try:\\n            return _PILE_TO_LABEL[str(rec[\\"pile\\"]).strip().lower()]\\n        except KeyError as exc:\\n            raise ValueError(f\\"unrecognized pile value {rec[\'pile\']!r}\\") from exc\\n    return int(default)\\n", "src/slopdet/lfm.py": "\\"\\"\\"LFM2 bidirectional encoder loading.\\n\\nThe trap: `AutoModel.from_pretrained(\\"LiquidAI/LFM2.5-Encoder-230M\\", trust_remote_code=True)` \\u2014 the\\nsnippet the model card gives for downstream heads \\u2014 returns an `Lfm2BidirectionalModel` whose weights\\nare **all freshly initialized**. The checkpoint stores body tensors under the `lfm2.` prefix, so the\\nbase-model class reports every parameter missing and every checkpoint key unexpected, then hands back a\\nrandom encoder that trains and evaluates without ever erroring.\\n\\nLoad the masked-LM wrapper (which owns the `lfm2.` prefix) and take its body instead, and assert the\\nload was clean so this can never regress silently.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\ntry:\\n    import torch  # noqa: F401\\n    from transformers import AutoModelForMaskedLM\\nexcept ImportError as exc:  # pragma: no cover\\n    raise ImportError(\\"lfm.py needs torch + transformers. Install the train extra.\\") from exc\\n\\n\\ndef load_encoder_body(model_name: str, dtype=None):\\n    \\"\\"\\"Return a fully-loaded LFM2 bidirectional encoder body.\\n\\n    Raises if any body tensor was initialized from scratch, which is what happens when the wrong\\n    auto-class is used.\\n    \\"\\"\\"\\n    mlm, info = AutoModelForMaskedLM.from_pretrained(\\n        model_name, trust_remote_code=True, dtype=dtype, output_loading_info=True\\n    )\\n    missing = [key for key in info.get(\\"missing_keys\\", []) if not key.startswith(\\"lm_head\\")]\\n    if missing:\\n        raise SystemExit(\\n            f\\"{model_name}: {len(missing)} body tensors were newly initialized \\"\\n            f\\"(first: {missing[:3]}). Refusing to train on a random encoder.\\"\\n        )\\n    return mlm.lfm2\\n", "ontology/schema.json": "{\\n  \\"$schema\\": \\"https://json-schema.org/draft/2020-12/schema\\",\\n  \\"$id\\": \\"https://slopdet.local/ontology/schema.json\\",\\n  \\"title\\": \\"Slop pattern\\",\\n  \\"description\\": \\"One frozen ontology entry. Ids are append-only. Disable with enabled=false; never reuse a deleted id.\\",\\n  \\"type\\": \\"object\\",\\n  \\"additionalProperties\\": false,\\n  \\"required\\": [\\n    \\"id\\",\\n    \\"lane\\",\\n    \\"unit\\",\\n    \\"detector\\",\\n    \\"pattern\\",\\n    \\"fix\\",\\n    \\"source\\",\\n    \\"license\\",\\n    \\"min_len_words\\",\\n    \\"paper\\",\\n    \\"enabled\\"\\n  ],\\n  \\"properties\\": {\\n    \\"id\\": {\\n      \\"type\\": \\"string\\",\\n      \\"pattern\\": \\"^[a-z][a-z0-9_]*$\\",\\n      \\"description\\": \\"Stable snake_case id. Never reused.\\"\\n    },\\n    \\"lane\\": {\\n      \\"type\\": \\"string\\",\\n      \\"enum\\": [\\"style\\", \\"rhetorical\\", \\"construction\\"]\\n    },\\n    \\"unit\\": {\\n      \\"type\\": \\"string\\",\\n      \\"enum\\": [\\"span\\", \\"sentence\\", \\"paragraph\\", \\"piece\\"]\\n    },\\n    \\"detector\\": {\\n      \\"type\\": \\"string\\",\\n      \\"enum\\": [\\"regex\\", \\"heuristic\\", \\"model_only\\"]\\n    },\\n    \\"pattern\\": {\\n      \\"type\\": \\"string\\",\\n      \\"minLength\\": 1,\\n      \\"description\\": \\"regex module pattern. Heuristic/model_only entries still ship a compiling proxy.\\"\\n    },\\n    \\"fix\\": {\\n      \\"type\\": \\"string\\",\\n      \\"minLength\\": 8,\\n      \\"description\\": \\"Few-word rewrite instruction. A pattern with no fix is not evidence.\\"\\n    },\\n    \\"source\\": {\\n      \\"type\\": \\"string\\",\\n      \\"minLength\\": 1\\n    },\\n    \\"license\\": {\\n      \\"type\\": \\"string\\",\\n      \\"enum\\": [\\"MIT-compatible\\", \\"CC-BY-SA-4.0\\", \\"Apache-2.0\\"]\\n    },\\n    \\"min_len_words\\": {\\n      \\"type\\": \\"integer\\",\\n      \\"minimum\\": 0\\n    },\\n    \\"paper\\": {\\n      \\"type\\": [\\"string\\", \\"null\\"]\\n    },\\n    \\"enabled\\": {\\n      \\"type\\": \\"boolean\\"\\n    }\\n  }\\n}\\n", "ontology/patterns.core.yaml": "- id: binary_contrast\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \'(?i)\\\\b(?:it(?:[\'\'\\u2019]s| is) not(?: just| only)?.{0,80}?\\\\b(?:it[\'\'\\u2019]s|it is|but)\\\\b|not (?:just|only)\\\\b.{0,60}?\\\\bbut(?: also)?\\\\b|the question isn[\'\'\\u2019]?t\\\\b.{0,60}?\\\\bit[\'\'\\u2019]s\\\\b)\'\\n  fix: State Y directly. Drop the not-X-but-Y frame.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: throat_clearing\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\b(?:here[\'\\u2019]?s the thing|here[\'\\u2019]?s what I mean|let me be clear|I[\'\\u2019]ll be honest|the uncomfortable truth is|here[\'\\u2019]?s the deal)\\\\b\\n  fix: Cut the opener. Start on the point.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: faux_insight\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\b(?:this is the part most people skip|what most people get wrong|here[\'\\u2019]s what nobody tells you|the part everyone misses|what nobody tells you)\\\\b\\n  fix: Drop the lone-expert setup. Make the claim stand alone.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: colon_reveal\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \'(?m)^[A-Z][^.!?\\\\n]{2,60}: [a-z]\'\\n  fix: Rewrite as a plain sentence.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: superficial_analysis\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i),\\\\s+(?:highlighting|underscoring|reflecting|showcasing|emphasizing|ensuring|symbolizing|demonstrating)\\\\b\\n  fix: Replace the trailing -ing clause with a concrete consequence.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: importance_puffery\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\b(?:stands as a testament|marks a pivotal moment|plays a vital role|solidifies its position|underscores its significance|crucial role|key turning point)\\\\b\\n  fix: State the fact. Let the reader judge if it matters.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: interpretive_metadiscourse\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \'(?i)\\\\b(?:that last part matters(?: more than it sounds)?|the key point is|as you can see|this distinction matters|in other words)\\\\b\'\\n  fix: Delete the aside, or replace it with a fact.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: weasel_attribution\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\b(?:experts agree|industry reports suggest|many argue|widely regarded as|studies show|observers have cited|some critics argue)\\\\b\\n  fix: Name the source or cut the claim.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: fake_strong_verb\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\b(?:serves as a|functions as a|acts as a|operates as a|stands as a)\\\\b\\n  fix: Use is/has, or name the actual action.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: synonym_cycling\\n  lane: style\\n  unit: sentence\\n  detector: heuristic\\n  pattern: (?i)\\\\b(?:the agent|the assistant|the tool|the system|the platform)\\\\b.{0,180}\\\\b(?:the agent|the assistant|the tool|the system|the platform)\\\\b\\n  fix: Repeat the clear word. Do not rotate synonyms for style.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: negative_listing\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bnot a\\\\b.{0,50}\\\\bnot a\\\\b.{0,50}\\\\b(?:a |an |the )\\n  fix: Just say Z. Drop the not-X not-Y list.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: dramatic_fragmentation\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)(?:\\\\bthat[\'\\u2019]s it\\\\. that[\'\\u2019]s the whole thing\\\\b|(?m)^And [A-Z][^.!?\\\\n]{0,50}\\\\.\\\\s*\\\\nAnd )\\n  fix: Use complete sentences. Stop stacking And-fragments.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: robotic_rhythm\\n  lane: style\\n  unit: paragraph\\n  detector: heuristic\\n  pattern: (?m)(?:^[A-Z][^.!?\\\\n]{10,42}\\\\.\\\\s+){2}[A-Z][^.!?\\\\n]{10,42}\\\\.\\n  fix: Vary sentence length and shape. Merge or split one of the three.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 40\\n  paper: null\\n  enabled: true\\n- id: rhetorical_setup\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\b(?:what if I told you|think about it:|plot twist:)\\\\b\\n  fix: Drop the setup. Make the point.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: fake_profound_kicker\\n  lane: style\\n  unit: sentence\\n  detector: regex\\n  pattern: (?i)\\\\b(?:and that(?:[\'\\u2019]s| is) the (?:whole |real )?point|the rest is (?:just )?noise|that(?:[\'\\u2019]s| is) the whole game)\\\\b\\n  fix: Delete the mic-drop. End on the last concrete sentence.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: recap_ending\\n  lane: style\\n  unit: paragraph\\n  detector: regex\\n  pattern: (?im)(?:^|(?<=[.!?]\\\\s))(?:in conclusion|ultimately|overall|to sum up|in summary|to conclude)\\\\s*[,:]\\n  fix: End on the last concrete point. Do not restate the piece.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: formatting_slop\\n  lane: style\\n  unit: paragraph\\n  detector: regex\\n  pattern: (?m)^#{1,3}\\\\s+.+\\\\n(?:.*\\\\n){0,2}^#{1,3}\\\\s+\\n  fix: Drop headers over two-sentence sections. Write prose.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: em_dash_cluster\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\u2014[^.\\\\n]{0,90}\\u2014\\n  fix: Use a comma, period, or parentheses. Do not cluster dashes.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: ban_delve_class\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\b(?:delve|delves|delving|foster|fostering|leverage|leveraging|utilize|utilizing|facilitate|facilitating|empower|empowering|streamline|streamlining)\\\\b\\n  fix: Name the action. Use a concrete verb.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: kobak-2406.07016\\n  enabled: true\\n- id: ban_puffery_noun\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \'(?i)\\\\b(?:tapestry|realm|beacon|paradigm(?: shift)?)\\\\b\'\\n  fix: Replace the metaphor with the actual thing.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: kobak-2406.07016\\n  enabled: true\\n- id: ban_puffery_adj\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\b(?:robust|cutting-edge|multifaceted|meticulous(?:ly)?|intricate|intricacies|paramount|transformative|vibrant|pivotal|groundbreaking|seamless(?:ly)?)\\\\b\\n  fix: Cut the adjective, or name the property it stands in for.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: kobak-2406.07016\\n  enabled: true\\n- id: ban_journey_verb\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\b(?:elevate|elevating|embark|embarking|supercharge|supercharging|harness|harnessing|ever-evolving)\\\\b\\n  fix: Use a plain verb. Say what actually happens.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: kobak-2406.07016\\n  enabled: true\\n- id: ban_showcase_verb\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\b(?:underscore|underscores|underscoring|showcase|showcases|showcasing|highlight|highlights|highlighting|emphasize|emphasizes|emphasizing)\\\\b\\n  fix: State the fact. Do not announce its importance.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: kobak-2406.07016\\n  enabled: true\\n- id: ban_corporate\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\b(?:synergy|synergies|pain points?|value proposition|thought leaders?(?:hip)?|circle back|touch base|move the needle)\\\\b\\n  fix: Say the work in ordinary words.\\n  source: anti-ai-slop-writing\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: ban_game_changer\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\b(?:game[- ]chang(?:er|ing)|this is huge|this changes everything|unlock(?:s|ing)? the power)\\\\b\\n  fix: Name the change. Drop the slogan.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: empty_adverb\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\b(?:literally|honestly|simply|actually|truly|fundamentally|importantly|crucially|inherently|inevitably)\\\\b\\n  fix: Cut the adverb if it adds nothing.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_worth_noting\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bit[\'\\u2019]?s worth noting\\\\b\\n  fix: Delete. Start with the fact.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_important_to_note\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \'(?i)\\\\bit[\'\'\\u2019]?s important to note(?: that)?\\\\b\'\\n  fix: Delete. State the fact.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_end_of_the_day\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bat the end of the day\\\\b\\n  fix: Cut the proverb. Make the claim.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_when_it_comes_to\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bwhen it comes to\\\\b\\n  fix: Name the subject and start.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_at_its_core\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bat its core\\\\b\\n  fix: Drop the frame. State the mechanism.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_in_todays\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bin today[\'\\u2019]?s\\\\b\\n  fix: Cut the era opener. Name the situation.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_in_the_age_of\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bin the age of\\\\b\\n  fix: Cut the era opener.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_in_the_world_of\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bin the world of\\\\b\\n  fix: Name the field. Skip the tour.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_the_reality_is\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bthe reality is\\\\b\\n  fix: Drop the drumroll. State the fact.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_the_truth_is\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bthe truth is\\\\b\\n  fix: Drop the drumroll. State the fact.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_in_terms_of\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bin terms of\\\\b\\n  fix: Rewrite with a direct object.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_with_regard_to\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bwith regard to\\\\b\\n  fix: Name the topic and continue.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_in_order_to\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bin order to\\\\b\\n  fix: Rewrite as \'to\' plus the verb.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_going_forward\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bgoing forward\\\\b\\n  fix: Cut it, or name the date.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_in_this_article\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bin this article\\\\b\\n  fix: Do not announce the article.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_lets_dive_in\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\blet[\'\\u2019]?s dive (?:in|deeper|into)\\\\b\\n  fix: Start the first fact.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_unlock_the_power\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bunlock(?:s|ing)? the power of\\\\b\\n  fix: Name the action the reader can take.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_bridge_the_gap\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bbridge(?:s|ing)? the gap\\\\b\\n  fix: Name the two sides and the actual link.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_i_hope_this_helps\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bI hope this helps\\\\b\\n  fix: End on the last useful sentence.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_i_hope_this_finds_you\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \'(?i)\\\\bI hope this(?: email)? finds you well\\\\b\'\\n  fix: Open with the reason you wrote.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_whether_youre\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bwhether you(?:[\'\\u2019]re| are) a\\\\b.{0,40}\\\\bor a\\\\b\\n  fix: Pick one reader. Write to them.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_from_x_to_y_opener\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)^From .{2,40} to .{2,40}[,.]\\n  fix: Open with the specific case, not a range.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_this_is_where_x_comes_in\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bthis is where\\\\b.{0,40}\\\\bcomes in\\\\b\\n  fix: Introduce the thing without the drumroll.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_firstly_secondly_thirdly\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\b(?:firstly|secondly|thirdly)\\\\b\\n  fix: Use 1. 2. 3. or just write the points.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_without_further_ado\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bwithout further ado\\\\b\\n  fix: Cut the drumroll and start.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_in_a_nutshell\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bin a nutshell\\\\b\\n  fix: State the summary as a sentence.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_please_dont_hesitate\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bplease don[\'\\u2019]?t hesitate to (?:reach out|contact)\\\\b\\n  fix: Give the actual next step.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_not_just_x_but_y\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bit[\'\\u2019]?s not just about\\\\b.{0,50}\\\\bit[\'\\u2019]?s about\\\\b\\n  fix: State Y. Drop the not-just frame.\\n  source: anti-ai-slop-writing\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: rule_of_three\\n  lane: style\\n  unit: sentence\\n  detector: heuristic\\n  pattern: (?i)\\\\b\\\\w+,\\\\s+\\\\w+,\\\\s+and\\\\s+\\\\w+\\\\b\\n  fix: Break the default trio. Use two, four, or one.\\n  source: anti-ai-slop-writing\\n  license: MIT-compatible\\n  min_len_words: 20\\n  paper: null\\n  enabled: true\\n- id: uniform_sentence_length\\n  lane: construction\\n  unit: paragraph\\n  detector: heuristic\\n  pattern: (?s)(?=.{80,})\\n  fix: Mix a short sentence with a long one.\\n  source: anti-ai-slop-writing\\n  license: MIT-compatible\\n  min_len_words: 80\\n  paper: null\\n  enabled: true\\n- id: parataxis\\n  lane: style\\n  unit: paragraph\\n  detector: heuristic\\n  pattern: (?m)(?:^[A-Z][^.!?\\\\n]{0,28}\\\\.\\\\s+){2}[A-Z][^.!?\\\\n]{0,28}\\\\.\\n  fix: Connect the thoughts. Add a because, but, or which.\\n  source: anti-ai-slop-writing\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: hedging_seesaw\\n  lane: style\\n  unit: paragraph\\n  detector: heuristic\\n  pattern: (?i)\\\\bon the one hand\\\\b.{0,200}\\\\bon the other hand\\\\b\\n  fix: Pick a side. Give the counterpoint one sentence.\\n  source: anti-ai-slop-writing\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: corporate_pep_talk\\n  lane: style\\n  unit: sentence\\n  detector: regex\\n  pattern: (?i)\\\\b(?:together we can|exciting opportunity|passionate about|unlock(?:ing)? potential|drive(?:s|ing)? (?:impact|outcomes)|deliver(?:ing)? value)\\\\b\\n  fix: Write like someone who did the work, including the mess.\\n  source: anti-ai-slop-writing\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: identical_paragraph_structure\\n  lane: construction\\n  unit: piece\\n  detector: heuristic\\n  pattern: (?s)(?=.{200,})\\n  fix: Break the topic-explain-example-transition mold.\\n  source: anti-ai-slop-writing\\n  license: MIT-compatible\\n  min_len_words: 200\\n  paper: null\\n  enabled: true\\n- id: bullet_overuse\\n  lane: style\\n  unit: paragraph\\n  detector: regex\\n  pattern: (?m)(?:^[\\\\t ]*(?:[-*]|\\\\d+\\\\.)\\\\s+.+\\\\n){6,}\\n  fix: Turn the list into sentences, or cap it at five.\\n  source: anti-ai-slop-writing\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: as_role_opener\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)^As an? [A-Z][^,.]{2,40},\\\\s+I\\\\b\\n  fix: Say the thing. Do not announce credentials.\\n  source: anti-ai-slop-writing\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: cross_section_parallelism\\n  lane: construction\\n  unit: piece\\n  detector: heuristic\\n  pattern: (?s)(?=.{300,})\\n  fix: Give each section a different shape and length.\\n  source: anti-ai-slop-writing\\n  license: MIT-compatible\\n  min_len_words: 300\\n  paper: null\\n  enabled: true\\n- id: passive_construction\\n  lane: style\\n  unit: sentence\\n  detector: heuristic\\n  pattern: (?i)\\\\b(?:is being \\\\w+ed|was found to be|are considered to be|has been shown to)\\\\b\\n  fix: Write the actor and the verb.\\n  source: anti-ai-slop-writing\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: mandatory_paragraph_transition\\n  lane: style\\n  unit: sentence\\n  detector: regex\\n  pattern: (?i)(?:^|\\\\n)\\\\s*(?:moreover|furthermore|additionally|in addition|that said|with that in mind)\\\\s*,\\n  fix: Let some paragraphs just stop.\\n  source: anti-ai-slop-writing\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: punct_em_dash_budget\\n  lane: style\\n  unit: piece\\n  detector: heuristic\\n  pattern: (?:\\u2014.*){2,}\\n  fix: At most one em dash per 500 words.\\n  source: anti-ai-slop-writing\\n  license: MIT-compatible\\n  min_len_words: 1\\n  paper: null\\n  enabled: true\\n- id: punct_exclamation_budget\\n  lane: style\\n  unit: piece\\n  detector: heuristic\\n  pattern: (?:!.*){2,}\\n  fix: At most one exclamation per 1,000 words.\\n  source: anti-ai-slop-writing\\n  license: MIT-compatible\\n  min_len_words: 1\\n  paper: null\\n  enabled: true\\n- id: punct_ellipsis_budget\\n  lane: style\\n  unit: piece\\n  detector: heuristic\\n  pattern: (?:\\\\.{3}|\\u2026).*(?:\\\\.{3}|\\u2026)\\n  fix: One ellipsis per piece, only for a real trailing-off.\\n  source: anti-ai-slop-writing\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: opener_certainly\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?im)^(?:certainly|absolutely|sure|great question|that[\'\\u2019]s a great point)[,!]\\n  fix: Answer. Skip the cheer.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: opener_moreover\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?im)^Moreover,\\n  fix: Start with the next fact.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: opener_furthermore\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?im)^Furthermore,\\n  fix: Start with the next fact.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: opener_additionally\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?im)^Additionally,\\n  fix: Start with the next fact.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: kobak-2406.07016\\n  enabled: true\\n- id: opener_interestingly\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?im)^Interestingly,\\n  fix: State the interesting fact. Drop the label.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: opener_notably\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?im)^Notably,\\n  fix: State the fact. Drop the label.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: opener_importantly\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?im)^Importantly,\\n  fix: State the fact. Drop the label.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: opener_indeed\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?im)^Indeed,\\n  fix: Continue without the nod.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: opener_as_an_ai\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\b(?:as an AI|as a language model)\\\\b\\n  fix: Never announce the model.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: emoji_bullet\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?m)^[\\\\t ]*[\\u2705\\ud83d\\udd25\\u2728\\ud83d\\udca1\\ud83d\\udc49\\u2b50\\ufe0f\\u2b50]\\\\s\\n  fix: Write a sentence. Do not emoji-bullet.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: hashtag_stack\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?:#[A-Za-z0-9_]+)(?:\\\\s+#[A-Za-z0-9_]+){2,}\\n  fix: Zero to two hashtags, in the sentence.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: markdown_in_plain\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?m)^\\\\*\\\\*[^*]{3,40}\\\\*\\\\*\\\\s*$\\n  fix: Do not bold a whole line for emphasis.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: copula_avoidance_surface\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\b(?:serves as|stands as|functions as|operates as|boasts a|features a)\\\\b\\n  fix: Use is or has when that is what you mean.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: reinhart-2410.16107\\n  enabled: true\\n- id: in_the_realm_of\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bin the realm of\\\\b\\n  fix: Name the field.\\n  source: anti-ai-slop-writing\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: a_testament_to\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\ba testament to\\\\b\\n  fix: State what happened.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: kobak-2406.07016\\n  enabled: true\\n- id: rest_assured\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\brest assured\\\\b\\n  fix: Give the actual guarantee or drop it.\\n  source: anti-ai-slop-writing\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: it_goes_without_saying\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bit goes without saying\\\\b\\n  fix: If it goes without saying, delete it.\\n  source: anti-ai-slop-writing\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: in_essence\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bin essence\\\\b\\n  fix: State the claim.\\n  source: anti-ai-slop-writing\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: please_note_that\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bplease note that\\\\b\\n  fix: State the note as a fact.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: as_mentioned_earlier\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bas (?:mentioned|noted|discussed) earlier\\\\b\\n  fix: Repeat the fact if needed. Do not point at the earlier sentence.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: in_todays_digital_age\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bin today[\'\\u2019]?s (?:fast-paced |ever-changing |digital )?world\\\\b\\n  fix: Name the actual constraint.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n", "ontology/patterns.wikipedia.yaml": "# SPDX-License-Identifier: CC-BY-SA-4.0\\n#\\n# Derived from Wikipedia:Signs of AI writing\\n# https://en.wikipedia.org/wiki/Wikipedia:Signs_of_AI_writing\\n# License: CC BY-SA 4.0\\n# https://creativecommons.org/licenses/by-sa/4.0/\\n#\\n# Share-alike applies to the descriptive text (fix blurbs) in this file.\\n# Regex strings are functional. Do not paste these descriptions into Apache-2.0 source.\\n#\\n- id: wiki_significance_puffery\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\b(?:stands as a testament|marking a pivotal moment|reflects broader|symbolizing its (?:ongoing|enduring|lasting)|setting the stage for|indelible mark|evolving landscape|focal point|deeply rooted)\\\\b\\n  fix: Cut the legacy sermon. Keep the dated fact.\\n  source: wikipedia-signs-of-ai-writing\\n  license: CC-BY-SA-4.0\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: wiki_notability_boilerplate\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\b(?:independent coverage|active social media presence|profiled in|widely-read outlets|significant, substantial, secondary coverage)\\\\b\\n  fix: Cite the source. Do not recite notability policy.\\n  source: wikipedia-signs-of-ai-writing\\n  license: CC-BY-SA-4.0\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: wiki_promotional_tone\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\b(?:nestled (?:in|within)|in the heart of|rich (?:cultural )?heritage|natural beauty|diverse array|boasts a|renowned for)\\\\b\\n  fix: Drop the brochure. Name one specific place or fact.\\n  source: wikipedia-signs-of-ai-writing\\n  license: CC-BY-SA-4.0\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: wiki_ai_vocabulary_cluster\\n  lane: style\\n  unit: paragraph\\n  detector: heuristic\\n  pattern: (?i)(?:\\\\b(?:delve|tapestry|underscore|pivotal|vibrant|intricate|meticulous|landscape|testament|showcase|foster|align with)\\\\b.*){3,}\\n  fix: One inflated word can be accident. Three in one passage is the tell. Rewrite with plain nouns.\\n  source: wikipedia-signs-of-ai-writing\\n  license: CC-BY-SA-4.0\\n  min_len_words: 0\\n  paper: kobak-2406.07016\\n  enabled: true\\n- id: wiki_copula_avoidance\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\b(?:serves as|stands as|marks a|functions as|operates as|holds the distinction of being|refers to)\\\\b\\n  fix: Use is or are. Stop dressing the copula.\\n  source: wikipedia-signs-of-ai-writing\\n  license: CC-BY-SA-4.0\\n  min_len_words: 0\\n  paper: reinhart-2410.16107\\n  enabled: true\\n- id: wiki_negative_parallelism\\n  lane: style\\n  unit: sentence\\n  detector: regex\\n  pattern: \'(?i)\\\\b(?:not only .{0,40} but(?: also)?|it is not .{0,40}, it(?:[\'\'\\u2019]s| is)|rather than .{0,30}$)\'\\n  fix: Drop the misconception-clearing frame. State the property.\\n  source: wikipedia-signs-of-ai-writing\\n  license: CC-BY-SA-4.0\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: wiki_challenges_future\\n  lane: style\\n  unit: paragraph\\n  detector: regex\\n  pattern: (?i)\\\\b(?:challenges remain|future prospects|looking ahead|as .+ continues to evolve|more research is needed)\\\\b\\n  fix: Stop the outline close. End on what is known now.\\n  source: wikipedia-signs-of-ai-writing\\n  license: CC-BY-SA-4.0\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: wiki_awards_heading\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)^#+\\\\s+Awards and recognition\\\\s*$\\n  fix: Merge awards into the career section if they are few.\\n  source: wikipedia-signs-of-ai-writing\\n  license: CC-BY-SA-4.0\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: wiki_title_heading\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?m)^#\\\\s+[A-Z].{0,80}\\\\n\\\\n\\n  fix: Do not repeat the article title as the first heading.\\n  source: wikipedia-signs-of-ai-writing\\n  license: CC-BY-SA-4.0\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: wiki_title_case_heading\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?m)^#{2,3}\\\\s+(?:[A-Z][a-z]+\\\\s+){2,}[A-Z][a-z]+$\\n  fix: Use sentence case in headings.\\n  source: wikipedia-signs-of-ai-writing\\n  license: CC-BY-SA-4.0\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: wiki_boldface_overuse\\n  lane: style\\n  unit: paragraph\\n  detector: regex\\n  pattern: (?:\\\\*\\\\*[^*]{2,40}\\\\*\\\\*.*){3,}\\n  fix: Bold once, if at all. Stop sprinkling emphasis.\\n  source: wikipedia-signs-of-ai-writing\\n  license: CC-BY-SA-4.0\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: wiki_inline_header_list\\n  lane: style\\n  unit: paragraph\\n  detector: regex\\n  pattern: (?m)^[-*]\\\\s+\\\\*\\\\*[^*]{2,40}\\\\*\\\\*:\\\\s\\n  fix: Turn canned bold-label bullets into sentences.\\n  source: wikipedia-signs-of-ai-writing\\n  license: CC-BY-SA-4.0\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: wiki_emoji_formatting\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?m)^[\\\\t ]*(?:[\\u2705\\u274c\\u26a0\\ufe0f\\ud83d\\udccc\\ud83d\\udd0d\\ud83d\\udca1]|:[a-z_]+:)\\\\s\\n  fix: No emoji as structure.\\n  source: wikipedia-signs-of-ai-writing\\n  license: CC-BY-SA-4.0\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: wiki_curly_quotes\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \'[\\u201c\\u201d\\u2018\\u2019].{0,80}[\\u201c\\u201d\\u2018\\u2019]\'\\n  fix: Straight quotes unless the house style needs curls.\\n  source: wikipedia-signs-of-ai-writing\\n  license: CC-BY-SA-4.0\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: wiki_collaborative_you\\n  lane: style\\n  unit: sentence\\n  detector: regex\\n  pattern: (?i)\\\\b(?:I hope this helps|let me know if|I[\'\\u2019]d be happy to|as you requested)\\\\b\\n  fix: This is article text, not a chat reply. Cut the assistant voice.\\n  source: wikipedia-signs-of-ai-writing\\n  license: CC-BY-SA-4.0\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: wiki_knowledge_cutoff\\n  lane: style\\n  unit: sentence\\n  detector: regex\\n  pattern: (?i)\\\\b(?:as of my last (?:training|update)|I don[\'\\u2019]t have (?:access|information)|my knowledge cutoff)\\\\b\\n  fix: Delete the model disclaimer. Write from sources.\\n  source: wikipedia-signs-of-ai-writing\\n  license: CC-BY-SA-4.0\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: wiki_placeholder_text\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\b(?:TODO|TBD|lorem ipsum|insert (?:text|citation|source) here|\\\\[placeholder\\\\])\\\\b\\n  fix: Replace placeholders before the text ships.\\n  source: wikipedia-signs-of-ai-writing\\n  license: CC-BY-SA-4.0\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: wiki_skipping_heading_levels\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?m)^#\\\\s+.+\\\\n+(?:#{3,}\\\\s+)\\n  fix: Do not skip from H1 to H3.\\n  source: wikipedia-signs-of-ai-writing\\n  license: CC-BY-SA-4.0\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: wiki_thematic_break_spam\\n  lane: style\\n  unit: piece\\n  detector: regex\\n  pattern: (?m)(?:^---\\\\s*\\\\n){2,}\\n  fix: Horizontal rules are not sectioning.\\n  source: wikipedia-signs-of-ai-writing\\n  license: CC-BY-SA-4.0\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: wiki_valuable_insights\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bvaluable insights\\\\b\\n  fix: Name the finding. Insights is empty.\\n  source: wikipedia-signs-of-ai-writing\\n  license: CC-BY-SA-4.0\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n", "ontology/patterns.rhetorical.yaml": "- id: rhet_present_participial\\n  lane: rhetorical\\n  unit: sentence\\n  detector: heuristic\\n  pattern: (?i),\\\\s+\\\\w+ing\\\\b.{0,80}(?:\\\\.|$)\\n  fix: The comma-VBG tag is a proxy for present-participial clauses. Replace with a finite clause.\\n  source: reinhart-biber\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: reinhart-2410.16107\\n  enabled: true\\n- id: rhet_nominalization_density\\n  lane: rhetorical\\n  unit: paragraph\\n  detector: heuristic\\n  pattern: (?i)\\\\b\\\\w{4,}(?:tion|sion|ment|ness|ity)s?\\\\b\\n  fix: High -tion/-ment/-ness/-ity rate. Prefer verbs over abstract nouns.\\n  source: reinhart-biber\\n  license: MIT-compatible\\n  min_len_words: 80\\n  paper: reinhart-2410.16107\\n  enabled: true\\n- id: rhet_copula_avoidance\\n  lane: rhetorical\\n  unit: paragraph\\n  detector: heuristic\\n  pattern: (?i)\\\\b(?:is|are|was|were|be|been|being)\\\\b\\n  fix: Low be-verb ratio vs lexical verbs is the tell. Restore is/are where they are clearer.\\n  source: reinhart-biber\\n  license: MIT-compatible\\n  min_len_words: 80\\n  paper: reinhart-2410.16107\\n  enabled: true\\n- id: rhet_that_complement\\n  lane: rhetorical\\n  unit: sentence\\n  detector: heuristic\\n  pattern: (?i)\\\\b(?:said|argued|claimed|noted|reported|suggested|found|showed|believed) that\\\\b\\n  fix: That-complement rate vs human baseline. Keep that when it prevents a garden path.\\n  source: reinhart-biber\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: reinhart-2410.16107\\n  enabled: true\\n- id: rhet_adj_stacking\\n  lane: rhetorical\\n  unit: span\\n  detector: heuristic\\n  pattern: (?i)\\\\b(?:a|an|the)\\\\s+[A-Za-z]+,\\\\s+[A-Za-z]+(?:,|\\\\s+and)\\\\s+[A-Za-z]+\\\\b\\n  fix: Stacked attributive adjectives. Keep one modifier that earns its place.\\n  source: reinhart-biber\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: reinhart-2410.16107\\n  enabled: true\\n", "ontology/patterns.slop.yaml": "# SPDX-License-Identifier: Apache-2.0\\n#\\n# Generated by scripts/emit_slop_ontology.py \\u2014 do not edit by hand.\\n# Source: sam-paech/antislop-sampler (Apache-2.0), ICLR 2026 antislop paper (arXiv:2510.15061).\\n# Phrases ranked by over-representation count in a large LLM-generated story dataset.\\n#\\n- id: slop_phrase_took_a_deep_breath\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\btook\\\\\\\\ a\\\\\\\\ deep\\\\\\\\ breath\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_voice_barely_above_a_whisper\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bvoice\\\\\\\\ barely\\\\\\\\ above\\\\\\\\ a\\\\\\\\ whisper\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_couldn_t_help_but_feel\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bcouldn\'t\\\\\\\\ help\\\\\\\\ but\\\\\\\\ feel\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_help_but_feel_a_sense\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bhelp\\\\\\\\ but\\\\\\\\ feel\\\\\\\\ a\\\\\\\\ sense\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_voice_barely_audible\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bvoice\\\\\\\\ barely\\\\\\\\ audible\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_casting_long_shadows\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bcasting\\\\\\\\ long\\\\\\\\ shadows\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_voice_barely_a_whisper\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bvoice\\\\\\\\ barely\\\\\\\\ a\\\\\\\\ whisper\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_couldn_t_shake_the_feeling\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bcouldn\'t\\\\\\\\ shake\\\\\\\\ the\\\\\\\\ feeling\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_couldn_t_help_but_wonder\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bcouldn\'t\\\\\\\\ help\\\\\\\\ but\\\\\\\\ wonder\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_long_shadows_across\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\blong\\\\\\\\ shadows\\\\\\\\ across\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_heart_pounding_in_my_chest\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bheart\\\\\\\\ pounding\\\\\\\\ in\\\\\\\\ my\\\\\\\\ chest\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_sun_dipped_below_the_horizon\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bsun\\\\\\\\ dipped\\\\\\\\ below\\\\\\\\ the\\\\\\\\ horizon\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_felt_a_chill_run\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bfelt\\\\\\\\ a\\\\\\\\ chill\\\\\\\\ run\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_air_was_thick_with_the_scent\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bair\\\\\\\\ was\\\\\\\\ thick\\\\\\\\ with\\\\\\\\ the\\\\\\\\ scent\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_felt_like_an_eternity\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bfelt\\\\\\\\ like\\\\\\\\ an\\\\\\\\ eternity\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_heart_pounding_in_her_chest\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bheart\\\\\\\\ pounding\\\\\\\\ in\\\\\\\\ her\\\\\\\\ chest\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_voice_steady_despite\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bvoice\\\\\\\\ steady\\\\\\\\ despite\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_felt_a_shiver_run\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bfelt\\\\\\\\ a\\\\\\\\ shiver\\\\\\\\ run\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_said_his_voice_low\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bsaid,\\\\\\\\ his\\\\\\\\ voice\\\\\\\\ low\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_room_fell_silent\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\broom\\\\\\\\ fell\\\\\\\\ silent\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_ready_to_face_whatever\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bready\\\\\\\\ to\\\\\\\\ face\\\\\\\\ whatever\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_trying_to_make_sense\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\btrying\\\\\\\\ to\\\\\\\\ make\\\\\\\\ sense\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_said_his_voice_barely\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bsaid,\\\\\\\\ his\\\\\\\\ voice\\\\\\\\ barely\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_dipped_below_the_horizon_casting\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bdipped\\\\\\\\ below\\\\\\\\ the\\\\\\\\ horizon,\\\\\\\\ casting\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_said_her_voice_barely\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bsaid,\\\\\\\\ her\\\\\\\\ voice\\\\\\\\ barely\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_asked_my_voice_barely\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\basked,\\\\\\\\ my\\\\\\\\ voice\\\\\\\\ barely\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_deep_breath_trying\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bdeep\\\\\\\\ breath,\\\\\\\\ trying\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_felt_a_strange_sense\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bfelt\\\\\\\\ a\\\\\\\\ strange\\\\\\\\ sense\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_something_else_entirely\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bsomething\\\\\\\\ else\\\\\\\\ entirely\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_could_feel_the_weight\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bcould\\\\\\\\ feel\\\\\\\\ the\\\\\\\\ weight\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_words_hung_in_the_air\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bwords\\\\\\\\ hung\\\\\\\\ in\\\\\\\\ the\\\\\\\\ air\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_heart_pounding_in_his_chest\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bheart\\\\\\\\ pounding\\\\\\\\ in\\\\\\\\ his\\\\\\\\ chest\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_brow_furrowed_in_concentration\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bbrow\\\\\\\\ furrowed\\\\\\\\ in\\\\\\\\ concentration\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_sun_began_to_set\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bsun\\\\\\\\ began\\\\\\\\ to\\\\\\\\ set\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_smile_playing_on_his_lips\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bsmile\\\\\\\\ playing\\\\\\\\ on\\\\\\\\ his\\\\\\\\ lips\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_voice_trembling_slightly\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bvoice\\\\\\\\ trembling\\\\\\\\ slightly\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_asked_her_voice_barely\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\basked,\\\\\\\\ her\\\\\\\\ voice\\\\\\\\ barely\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_door_creaked_open\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bdoor\\\\\\\\ creaked\\\\\\\\ open\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_eyes_never_leaving\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\beyes\\\\\\\\ never\\\\\\\\ leaving\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_days_turned_into_weeks\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bdays\\\\\\\\ turned\\\\\\\\ into\\\\\\\\ weeks\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_voice_a_low_rumble\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bvoice\\\\\\\\ a\\\\\\\\ low\\\\\\\\ rumble\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_growing_sense_of_unease\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bgrowing\\\\\\\\ sense\\\\\\\\ of\\\\\\\\ unease\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_took_a_step_back\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\btook\\\\\\\\ a\\\\\\\\ step\\\\\\\\ back\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_heart_skipped_a_beat\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bheart\\\\\\\\ skipped\\\\\\\\ a\\\\\\\\ beat\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_air_hung_thick\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bair\\\\\\\\ hung\\\\\\\\ thick\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_said_her_voice_steady\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bsaid,\\\\\\\\ her\\\\\\\\ voice\\\\\\\\ steady\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_rain_continued_to_fall\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\brain\\\\\\\\ continued\\\\\\\\ to\\\\\\\\ fall\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_sun_hung_low\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bsun\\\\\\\\ hung\\\\\\\\ low\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_shiver_run_down_my_spine\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bshiver\\\\\\\\ run\\\\\\\\ down\\\\\\\\ my\\\\\\\\ spine\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_took_a_step_forward\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\btook\\\\\\\\ a\\\\\\\\ step\\\\\\\\ forward\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_said_my_voice_barely\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bsaid,\\\\\\\\ my\\\\\\\\ voice\\\\\\\\ barely\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_casting_a_warm_glow\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bcasting\\\\\\\\ a\\\\\\\\ warm\\\\\\\\ glow\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_renewed_sense_of_purpose\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\brenewed\\\\\\\\ sense\\\\\\\\ of\\\\\\\\ purpose\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_spreading_across_his_face\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bspreading\\\\\\\\ across\\\\\\\\ his\\\\\\\\ face\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_taking_a_deep_breath\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bTaking\\\\\\\\ a\\\\\\\\ deep\\\\\\\\ breath\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_horizon_casting_long\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bhorizon,\\\\\\\\ casting\\\\\\\\ long\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_hung_low_in_the_sky\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bhung\\\\\\\\ low\\\\\\\\ in\\\\\\\\ the\\\\\\\\ sky\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_whispered_her_voice_barely\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bwhispered,\\\\\\\\ her\\\\\\\\ voice\\\\\\\\ barely\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_smile_spreading_across\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bsmile\\\\\\\\ spreading\\\\\\\\ across\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_leaned_back_in_his_chair\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bleaned\\\\\\\\ back\\\\\\\\ in\\\\\\\\ his\\\\\\\\ chair\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_low_in_the_sky_casting\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\blow\\\\\\\\ in\\\\\\\\ the\\\\\\\\ sky,\\\\\\\\ casting\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_hung_heavy_in_the_air\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bhung\\\\\\\\ heavy\\\\\\\\ in\\\\\\\\ the\\\\\\\\ air\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_eyes_wide_with_fear\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\beyes\\\\\\\\ wide\\\\\\\\ with\\\\\\\\ fear\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_took_a_step_closer\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\btook\\\\\\\\ a\\\\\\\\ step\\\\\\\\ closer\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_shake_the_feeling_that_something\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bshake\\\\\\\\ the\\\\\\\\ feeling\\\\\\\\ that\\\\\\\\ something\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_something_else_something\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bsomething\\\\\\\\ else,\\\\\\\\ something\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_face_whatever_challenges\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bface\\\\\\\\ whatever\\\\\\\\ challenges\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_one_last_time\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bone\\\\\\\\ last\\\\\\\\ time\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_spread_like_wildfire\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bspread\\\\\\\\ like\\\\\\\\ wildfire\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_asked_his_voice_barely\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\basked,\\\\\\\\ his\\\\\\\\ voice\\\\\\\\ barely\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_road_ahead_would\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\broad\\\\\\\\ ahead\\\\\\\\ would\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_felt_a_sense_of_peace\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bfelt\\\\\\\\ a\\\\\\\\ sense\\\\\\\\ of\\\\\\\\ peace\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_newfound_sense_of_purpose\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bnewfound\\\\\\\\ sense\\\\\\\\ of\\\\\\\\ purpose\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_door_swung_open\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bdoor\\\\\\\\ swung\\\\\\\\ open\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_grin_spreading_across\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bgrin\\\\\\\\ spreading\\\\\\\\ across\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_eyes_filled_with_a_mixture\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\beyes\\\\\\\\ filled\\\\\\\\ with\\\\\\\\ a\\\\\\\\ mixture\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_said_his_voice_a_low\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bsaid,\\\\\\\\ his\\\\\\\\ voice\\\\\\\\ a\\\\\\\\ low\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_flicker_of_something_akin\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bflicker\\\\\\\\ of\\\\\\\\ something\\\\\\\\ akin\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_eyes_locked_onto\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\beyes\\\\\\\\ locked\\\\\\\\ onto\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_dimly_lit_room\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bdimly\\\\\\\\ lit\\\\\\\\ room\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_tried_to_make_sense\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\btried\\\\\\\\ to\\\\\\\\ make\\\\\\\\ sense\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_challenges_lay_ahead\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bchallenges\\\\\\\\ lay\\\\\\\\ ahead\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_hung_in_the_air_heavy\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bhung\\\\\\\\ in\\\\\\\\ the\\\\\\\\ air,\\\\\\\\ heavy\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_chill_run_down_my_spine\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bchill\\\\\\\\ run\\\\\\\\ down\\\\\\\\ my\\\\\\\\ spine\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_small_intricately_carved\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bsmall,\\\\\\\\ intricately\\\\\\\\ carved\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_said_his_voice_filled\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bsaid,\\\\\\\\ his\\\\\\\\ voice\\\\\\\\ filled\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_eyes_darting_around\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\beyes\\\\\\\\ darting\\\\\\\\ around\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_said_his_voice_steady\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bsaid,\\\\\\\\ his\\\\\\\\ voice\\\\\\\\ steady\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_couldn_t_help_but_notice\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bcouldn\'t\\\\\\\\ help\\\\\\\\ but\\\\\\\\ notice\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_deep_breath_steeling\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bdeep\\\\\\\\ breath,\\\\\\\\ steeling\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_brow_furrowed_in_confusion\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bbrow\\\\\\\\ furrowed\\\\\\\\ in\\\\\\\\ confusion\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_sent_a_shiver_down_my_spine\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bsent\\\\\\\\ a\\\\\\\\ shiver\\\\\\\\ down\\\\\\\\ my\\\\\\\\ spine\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_chill_run_down_her_spine\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bchill\\\\\\\\ run\\\\\\\\ down\\\\\\\\ her\\\\\\\\ spine\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_would_find_a_way\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bwould\\\\\\\\ find\\\\\\\\ a\\\\\\\\ way\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_young_woman_named\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\byoung\\\\\\\\ woman\\\\\\\\ named\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_breath_caught_in_her_throat\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bbreath\\\\\\\\ caught\\\\\\\\ in\\\\\\\\ her\\\\\\\\ throat\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_fingers_flying_across\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bfingers\\\\\\\\ flying\\\\\\\\ across\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_eyes_wide_with_wonder\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\beyes\\\\\\\\ wide\\\\\\\\ with\\\\\\\\ wonder\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_dust_motes_danced\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bDust\\\\\\\\ motes\\\\\\\\ danced\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_mind_raced_trying\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bmind\\\\\\\\ raced,\\\\\\\\ trying\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_figure_emerged_from_the_shadows\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bfigure\\\\\\\\ emerged\\\\\\\\ from\\\\\\\\ the\\\\\\\\ shadows\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_heart_hammered_against_my_ribs\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bheart\\\\\\\\ hammered\\\\\\\\ against\\\\\\\\ my\\\\\\\\ ribs\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_turned_and_walked_away\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bturned\\\\\\\\ and\\\\\\\\ walked\\\\\\\\ away\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_piercing_blue_eyes\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bpiercing\\\\\\\\ blue\\\\\\\\ eyes\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_felt_a_strange_sensation\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bfelt\\\\\\\\ a\\\\\\\\ strange\\\\\\\\ sensation\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_small_smile_playing\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bsmall\\\\\\\\ smile\\\\\\\\ playing\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_trying_to_keep_my_voice\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\btrying\\\\\\\\ to\\\\\\\\ keep\\\\\\\\ my\\\\\\\\ voice\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_felt_a_cold_dread\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bfelt\\\\\\\\ a\\\\\\\\ cold\\\\\\\\ dread\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_hung_thick_with_the_scent\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bhung\\\\\\\\ thick\\\\\\\\ with\\\\\\\\ the\\\\\\\\ scent\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_air_was_thick_with_tension\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bair\\\\\\\\ was\\\\\\\\ thick\\\\\\\\ with\\\\\\\\ tension\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_sky_casting_long\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bsky,\\\\\\\\ casting\\\\\\\\ long\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_would_never_forget\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bwould\\\\\\\\ never\\\\\\\\ forget\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_whatever_challenges_lay\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bwhatever\\\\\\\\ challenges\\\\\\\\ lay\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_mind_racing_with_questions\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bmind\\\\\\\\ racing\\\\\\\\ with\\\\\\\\ questions\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_said_trying_to_sound\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bsaid,\\\\\\\\ trying\\\\\\\\ to\\\\\\\\ sound\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_said_her_voice_trembling\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bsaid,\\\\\\\\ her\\\\\\\\ voice\\\\\\\\ trembling\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_gaze_sweeping_across\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bgaze\\\\\\\\ sweeping\\\\\\\\ across\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_spent_countless_hours\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bspent\\\\\\\\ countless\\\\\\\\ hours\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_said_his_voice_dripping\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bsaid,\\\\\\\\ his\\\\\\\\ voice\\\\\\\\ dripping\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_resonated_deep_within\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bresonated\\\\\\\\ deep\\\\\\\\ within\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_first_time_in_a_long\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bfirst\\\\\\\\ time\\\\\\\\ in\\\\\\\\ a\\\\\\\\ long\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_blood_ran_cold\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bblood\\\\\\\\ ran\\\\\\\\ cold\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_deep_breath_feeling\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bdeep\\\\\\\\ breath,\\\\\\\\ feeling\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_mind_racing_with_the_implications\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bmind\\\\\\\\ racing\\\\\\\\ with\\\\\\\\ the\\\\\\\\ implications\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_mind_racing_with_possibilities\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bmind\\\\\\\\ racing\\\\\\\\ with\\\\\\\\ possibilities\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_eyes_widened_in_surprise\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\beyes\\\\\\\\ widened\\\\\\\\ in\\\\\\\\ surprise\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_said_her_voice_filled\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bsaid,\\\\\\\\ her\\\\\\\\ voice\\\\\\\\ filled\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_eyes_wide_with_a_mixture\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\beyes\\\\\\\\ wide\\\\\\\\ with\\\\\\\\ a\\\\\\\\ mixture\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_smile_playing_on_her_lips\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bsmile\\\\\\\\ playing\\\\\\\\ on\\\\\\\\ her\\\\\\\\ lips\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_could_find_a_way\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bcould\\\\\\\\ find\\\\\\\\ a\\\\\\\\ way\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_never_seen_anything\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bnever\\\\\\\\ seen\\\\\\\\ anything\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_knew_one_thing\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bknew\\\\\\\\ one\\\\\\\\ thing\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_said_my_voice_steady\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bsaid,\\\\\\\\ my\\\\\\\\ voice\\\\\\\\ steady\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_air_thick_with_the_scent\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bair\\\\\\\\ thick\\\\\\\\ with\\\\\\\\ the\\\\\\\\ scent\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_eyes_scanning_the_room\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\beyes\\\\\\\\ scanning\\\\\\\\ the\\\\\\\\ room\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_felt_a_growing_sense\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bfelt\\\\\\\\ a\\\\\\\\ growing\\\\\\\\ sense\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_seen_anything_like\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bseen\\\\\\\\ anything\\\\\\\\ like\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_asked_her_voice_trembling\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\basked,\\\\\\\\ her\\\\\\\\ voice\\\\\\\\ trembling\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_help_but_feel_a_twinge\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bhelp\\\\\\\\ but\\\\\\\\ feel\\\\\\\\ a\\\\\\\\ twinge\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_smile_spread_across\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bsmile\\\\\\\\ spread\\\\\\\\ across\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_breath_caught_in_my_throat\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bbreath\\\\\\\\ caught\\\\\\\\ in\\\\\\\\ my\\\\\\\\ throat\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_heart_pounded_in_my_chest\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bheart\\\\\\\\ pounded\\\\\\\\ in\\\\\\\\ my\\\\\\\\ chest\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_feel_a_sense_of_unease\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bfeel\\\\\\\\ a\\\\\\\\ sense\\\\\\\\ of\\\\\\\\ unease\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_scent_of_damp_earth\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bscent\\\\\\\\ of\\\\\\\\ damp\\\\\\\\ earth\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_growing_sense_of_dread\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bgrowing\\\\\\\\ sense\\\\\\\\ of\\\\\\\\ dread\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_looked_around_the_room\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\blooked\\\\\\\\ around\\\\\\\\ the\\\\\\\\ room\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_intricately_carved_wooden\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bintricately\\\\\\\\ carved\\\\\\\\ wooden\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_raised_a_hand_silencing\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\braised\\\\\\\\ a\\\\\\\\ hand,\\\\\\\\ silencing\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_began_to_set_casting\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bbegan\\\\\\\\ to\\\\\\\\ set,\\\\\\\\ casting\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_sighed_running_a_hand\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bsighed,\\\\\\\\ running\\\\\\\\ a\\\\\\\\ hand\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_hand_instinctively_reaching\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bhand\\\\\\\\ instinctively\\\\\\\\ reaching\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_sense_of_peace_wash\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bsense\\\\\\\\ of\\\\\\\\ peace\\\\\\\\ wash\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_heart_heavy_with_the_weight\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bheart\\\\\\\\ heavy\\\\\\\\ with\\\\\\\\ the\\\\\\\\ weight\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_knew_that_the_road_ahead\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bknew\\\\\\\\ that\\\\\\\\ the\\\\\\\\ road\\\\\\\\ ahead\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_said_her_voice_soft\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bsaid,\\\\\\\\ her\\\\\\\\ voice\\\\\\\\ soft\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_smile_tugging_at_the_corners\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bsmile\\\\\\\\ tugging\\\\\\\\ at\\\\\\\\ the\\\\\\\\ corners\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_leaned_forward_his_eyes\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bleaned\\\\\\\\ forward,\\\\\\\\ his\\\\\\\\ eyes\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_keep_my_voice_steady\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bkeep\\\\\\\\ my\\\\\\\\ voice\\\\\\\\ steady\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_knuckles_turning_white\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bknuckles\\\\\\\\ turning\\\\\\\\ white\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_said_her_voice_firm\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bsaid,\\\\\\\\ her\\\\\\\\ voice\\\\\\\\ firm\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_felt_a_glimmer_of_hope\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bfelt\\\\\\\\ a\\\\\\\\ glimmer\\\\\\\\ of\\\\\\\\ hope\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_heart_pounded_in_her_chest\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bheart\\\\\\\\ pounded\\\\\\\\ in\\\\\\\\ her\\\\\\\\ chest\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_cast_long_shadows\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bcast\\\\\\\\ long\\\\\\\\ shadows\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_eyes_widened_in_shock\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\beyes\\\\\\\\ widened\\\\\\\\ in\\\\\\\\ shock\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_first_time_since\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bfirst\\\\\\\\ time\\\\\\\\ since\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_air_grew_thick\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bair\\\\\\\\ grew\\\\\\\\ thick\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_feel_a_sense_of_pride\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bfeel\\\\\\\\ a\\\\\\\\ sense\\\\\\\\ of\\\\\\\\ pride\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_horizon_painting_the_sky\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bhorizon,\\\\\\\\ painting\\\\\\\\ the\\\\\\\\ sky\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_whispered_his_voice_barely\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bwhispered,\\\\\\\\ his\\\\\\\\ voice\\\\\\\\ barely\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_faint_almost_imperceptible\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bfaint,\\\\\\\\ almost\\\\\\\\ imperceptible\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_said_his_voice_firm\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bsaid,\\\\\\\\ his\\\\\\\\ voice\\\\\\\\ firm\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_continued_to_fall_washing\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bcontinued\\\\\\\\ to\\\\\\\\ fall,\\\\\\\\ washing\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_casting_an_eerie_glow\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bcasting\\\\\\\\ an\\\\\\\\ eerie\\\\\\\\ glow\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_ahead_would_be_long\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bahead\\\\\\\\ would\\\\\\\\ be\\\\\\\\ long\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_knew_with_a_chilling_certainty\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bknew,\\\\\\\\ with\\\\\\\\ a\\\\\\\\ chilling\\\\\\\\ certainty\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_eyes_locking_onto\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\beyes\\\\\\\\ locking\\\\\\\\ onto\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_voice_thick_with_emotion\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bvoice\\\\\\\\ thick\\\\\\\\ with\\\\\\\\ emotion\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_mind_already_racing\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bmind\\\\\\\\ already\\\\\\\\ racing\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_air_was_thick_with_anticipation\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bair\\\\\\\\ was\\\\\\\\ thick\\\\\\\\ with\\\\\\\\ anticipation\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_said_trying_to_keep\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bsaid,\\\\\\\\ trying\\\\\\\\ to\\\\\\\\ keep\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_find_a_way_to_break\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bfind\\\\\\\\ a\\\\\\\\ way\\\\\\\\ to\\\\\\\\ break\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_long_dancing_shadows\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\blong,\\\\\\\\ dancing\\\\\\\\ shadows\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_uuwu_uuwu_uuwu\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\buuwu,\\\\\\\\ uuwu,\\\\\\\\ uuwu\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_casting_a_golden_glow\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bcasting\\\\\\\\ a\\\\\\\\ golden\\\\\\\\ glow\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_chill_run_down_his_spine\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bchill\\\\\\\\ run\\\\\\\\ down\\\\\\\\ his\\\\\\\\ spine\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_whispered_her_voice_trembling\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bwhispered,\\\\\\\\ her\\\\\\\\ voice\\\\\\\\ trembling\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_needed_to_find_a_way\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bneeded\\\\\\\\ to\\\\\\\\ find\\\\\\\\ a\\\\\\\\ way\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_change_the_course_of_history\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bchange\\\\\\\\ the\\\\\\\\ course\\\\\\\\ of\\\\\\\\ history\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_couldn_t_quite_place\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bcouldn\'t\\\\\\\\ quite\\\\\\\\ place\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_eyes_wide_with_terror\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\beyes\\\\\\\\ wide\\\\\\\\ with\\\\\\\\ terror\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_pushed_open_the_door\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bpushed\\\\\\\\ open\\\\\\\\ the\\\\\\\\ door\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_time_would_tell\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\btime\\\\\\\\ would\\\\\\\\ tell\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_would_change_the_course\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bwould\\\\\\\\ change\\\\\\\\ the\\\\\\\\ course\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_need_to_find_a_way\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bneed\\\\\\\\ to\\\\\\\\ find\\\\\\\\ a\\\\\\\\ way\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_sent_shivers_down_my_spine\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bsent\\\\\\\\ shivers\\\\\\\\ down\\\\\\\\ my\\\\\\\\ spine\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_asked_my_voice_trembling\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\basked,\\\\\\\\ my\\\\\\\\ voice\\\\\\\\ trembling\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_find_a_way_to_make\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bfind\\\\\\\\ a\\\\\\\\ way\\\\\\\\ to\\\\\\\\ make\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_painting_the_sky_in_hues\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bpainting\\\\\\\\ the\\\\\\\\ sky\\\\\\\\ in\\\\\\\\ hues\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_eyes_wide_with_disbelief\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\beyes\\\\\\\\ wide\\\\\\\\ with\\\\\\\\ disbelief\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_air_grew_colder\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bair\\\\\\\\ grew\\\\\\\\ colder\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_said_her_voice_low\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bsaid,\\\\\\\\ her\\\\\\\\ voice\\\\\\\\ low\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_time_in_a_long_time\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\btime\\\\\\\\ in\\\\\\\\ a\\\\\\\\ long\\\\\\\\ time\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_began_to_take_shape\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bbegan\\\\\\\\ to\\\\\\\\ take\\\\\\\\ shape\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_life_would_never\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\blife\\\\\\\\ would\\\\\\\\ never\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_said_his_voice_laced\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bsaid,\\\\\\\\ his\\\\\\\\ voice\\\\\\\\ laced\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_small_almost_imperceptible\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bsmall,\\\\\\\\ almost\\\\\\\\ imperceptible\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_eyes_filled_with_tears\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\beyes\\\\\\\\ filled\\\\\\\\ with\\\\\\\\ tears\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_one_thing_was_certain\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bone\\\\\\\\ thing\\\\\\\\ was\\\\\\\\ certain\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_challenges_that_lay_ahead\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bchallenges\\\\\\\\ that\\\\\\\\ lay\\\\\\\\ ahead\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_cool_night_air\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bcool\\\\\\\\ night\\\\\\\\ air\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_whatever_lay_ahead\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bwhatever\\\\\\\\ lay\\\\\\\\ ahead\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_could_feel_the_power\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bcould\\\\\\\\ feel\\\\\\\\ the\\\\\\\\ power\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_first_time_in_years\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bfirst\\\\\\\\ time\\\\\\\\ in\\\\\\\\ years\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_legs_over_the_side_of_the_bed\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\blegs\\\\\\\\ over\\\\\\\\ the\\\\\\\\ side\\\\\\\\ of\\\\\\\\ the\\\\\\\\ bed\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_one_step_ahead\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bone\\\\\\\\ step\\\\\\\\ ahead\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_eyes_fluttered_open\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\beyes\\\\\\\\ fluttered\\\\\\\\ open\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_shiver_ran_down_my_spine\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bshiver\\\\\\\\ ran\\\\\\\\ down\\\\\\\\ my\\\\\\\\ spine\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_like_a_physical_blow\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\blike\\\\\\\\ a\\\\\\\\ physical\\\\\\\\ blow\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_chill_ran_down_my_spine\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bchill\\\\\\\\ ran\\\\\\\\ down\\\\\\\\ my\\\\\\\\ spine\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_couldn_t_help_but_smile\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bcouldn\'t\\\\\\\\ help\\\\\\\\ but\\\\\\\\ smile\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_nodded_a_small_smile\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bnodded,\\\\\\\\ a\\\\\\\\ small\\\\\\\\ smile\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_set_casting_long\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bset,\\\\\\\\ casting\\\\\\\\ long\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_horizon_casting_a_warm\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bhorizon,\\\\\\\\ casting\\\\\\\\ a\\\\\\\\ warm\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_given_a_second_chance\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bgiven\\\\\\\\ a\\\\\\\\ second\\\\\\\\ chance\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_voice_tinged_with_a_hint\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bvoice\\\\\\\\ tinged\\\\\\\\ with\\\\\\\\ a\\\\\\\\ hint\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_shiver_run_down_her_spine\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bshiver\\\\\\\\ run\\\\\\\\ down\\\\\\\\ her\\\\\\\\ spine\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_took_another_step\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\btook\\\\\\\\ another\\\\\\\\ step\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_like_a_second_skin\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\blike\\\\\\\\ a\\\\\\\\ second\\\\\\\\ skin\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_make_things_right\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bmake\\\\\\\\ things\\\\\\\\ right\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_shook_my_head_trying\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bshook\\\\\\\\ my\\\\\\\\ head,\\\\\\\\ trying\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_asked_trying_to_keep\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\basked,\\\\\\\\ trying\\\\\\\\ to\\\\\\\\ keep\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_eyes_darted_around\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\beyes\\\\\\\\ darted\\\\\\\\ around\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_could_almost_hear\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bcould\\\\\\\\ almost\\\\\\\\ hear\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_tasting_like_ash\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\btasting\\\\\\\\ like\\\\\\\\ ash\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_darting_around_the_room\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bdarting\\\\\\\\ around\\\\\\\\ the\\\\\\\\ room\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_exchanged_uneasy_glances\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bexchanged\\\\\\\\ uneasy\\\\\\\\ glances\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_screen_flickered_to_life\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bscreen\\\\\\\\ flickered\\\\\\\\ to\\\\\\\\ life\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_whatever_came_next\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bwhatever\\\\\\\\ came\\\\\\\\ next\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_mix_of_excitement_and_trepidation\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bmix\\\\\\\\ of\\\\\\\\ excitement\\\\\\\\ and\\\\\\\\ trepidation\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_fingers_dancing_across\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bfingers\\\\\\\\ dancing\\\\\\\\ across\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_heart_pounding_with_a_mixture\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bheart\\\\\\\\ pounding\\\\\\\\ with\\\\\\\\ a\\\\\\\\ mixture\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_felt_like_hours\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bfelt\\\\\\\\ like\\\\\\\\ hours\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_felt_a_flicker_of_hope\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bfelt\\\\\\\\ a\\\\\\\\ flicker\\\\\\\\ of\\\\\\\\ hope\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_felt_a_surge_of_energy\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bfelt\\\\\\\\ a\\\\\\\\ surge\\\\\\\\ of\\\\\\\\ energy\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_blinding_flash_of_light\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bblinding\\\\\\\\ flash\\\\\\\\ of\\\\\\\\ light\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_heart_pounded_in_his_chest\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bheart\\\\\\\\ pounded\\\\\\\\ in\\\\\\\\ his\\\\\\\\ chest\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_wave_of_nausea_washed\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bwave\\\\\\\\ of\\\\\\\\ nausea\\\\\\\\ washed\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_senses_on_high_alert\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bsenses\\\\\\\\ on\\\\\\\\ high\\\\\\\\ alert\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_deep_breath_and_stepped\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bdeep\\\\\\\\ breath\\\\\\\\ and\\\\\\\\ stepped\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_blood_run_cold\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bblood\\\\\\\\ run\\\\\\\\ cold\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_brow_furrowed_with_concern\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bbrow\\\\\\\\ furrowed\\\\\\\\ with\\\\\\\\ concern\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_warm_golden_glow\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bwarm,\\\\\\\\ golden\\\\\\\\ glow\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_heart_skip_a_beat\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bheart\\\\\\\\ skip\\\\\\\\ a\\\\\\\\ beat\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_air_hung_heavy\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bair\\\\\\\\ hung\\\\\\\\ heavy\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_seemed_to_hold_its_breath\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bseemed\\\\\\\\ to\\\\\\\\ hold\\\\\\\\ its\\\\\\\\ breath\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_air_crackled_with_energy\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bair\\\\\\\\ crackled\\\\\\\\ with\\\\\\\\ energy\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_mind_racing_with_a_thousand\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bmind\\\\\\\\ racing\\\\\\\\ with\\\\\\\\ a\\\\\\\\ thousand\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_lumina_lumina_lumina\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bLumina,\\\\\\\\ Lumina,\\\\\\\\ Lumina\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_locked_onto_mine\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\blocked\\\\\\\\ onto\\\\\\\\ mine\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_felt_a_surge_of_anger\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bfelt\\\\\\\\ a\\\\\\\\ surge\\\\\\\\ of\\\\\\\\ anger\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_voice_devoid_of_emotion\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bvoice\\\\\\\\ devoid\\\\\\\\ of\\\\\\\\ emotion\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_first_light_of_dawn\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bfirst\\\\\\\\ light\\\\\\\\ of\\\\\\\\ dawn\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_breath_trying_to_steady\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bbreath,\\\\\\\\ trying\\\\\\\\ to\\\\\\\\ steady\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_leaned_back_in_her_chair\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bleaned\\\\\\\\ back\\\\\\\\ in\\\\\\\\ her\\\\\\\\ chair\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_world_held_its_breath\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bworld\\\\\\\\ held\\\\\\\\ its\\\\\\\\ breath\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_young_man_named\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\byoung\\\\\\\\ man\\\\\\\\ named\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_dancing_shadows_across\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bdancing\\\\\\\\ shadows\\\\\\\\ across\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_dimly_lit_chamber\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bdimly\\\\\\\\ lit\\\\\\\\ chamber\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_hung_in_the_air_like\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bhung\\\\\\\\ in\\\\\\\\ the\\\\\\\\ air\\\\\\\\ like\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_watched_in_stunned_silence\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bwatched\\\\\\\\ in\\\\\\\\ stunned\\\\\\\\ silence\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_course_of_human_history\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bcourse\\\\\\\\ of\\\\\\\\ human\\\\\\\\ history\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_steady_despite_the_turmoil\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bsteady\\\\\\\\ despite\\\\\\\\ the\\\\\\\\ turmoil\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_tall_imposing_figure\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\btall,\\\\\\\\ imposing\\\\\\\\ figure\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_strange_sense_of_peace\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bstrange\\\\\\\\ sense\\\\\\\\ of\\\\\\\\ peace\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_question_hung_in_the_air\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bquestion\\\\\\\\ hung\\\\\\\\ in\\\\\\\\ the\\\\\\\\ air\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_felt_a_sense_of_purpose\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bfelt\\\\\\\\ a\\\\\\\\ sense\\\\\\\\ of\\\\\\\\ purpose\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_would_do_whatever_it_took\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bwould\\\\\\\\ do\\\\\\\\ whatever\\\\\\\\ it\\\\\\\\ took\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_new_york_city\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bNew\\\\\\\\ York\\\\\\\\ City\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_breath_trying_to_calm\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bbreath,\\\\\\\\ trying\\\\\\\\ to\\\\\\\\ calm\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_sound_like_wind\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bsound\\\\\\\\ like\\\\\\\\ wind\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_said_his_voice_trembling\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bsaid,\\\\\\\\ his\\\\\\\\ voice\\\\\\\\ trembling\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_dipped_below_the_horizon_painting\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bdipped\\\\\\\\ below\\\\\\\\ the\\\\\\\\ horizon,\\\\\\\\ painting\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_faces_etched_with_a_mixture\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bfaces\\\\\\\\ etched\\\\\\\\ with\\\\\\\\ a\\\\\\\\ mixture\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_one_thing_was_clear\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bone\\\\\\\\ thing\\\\\\\\ was\\\\\\\\ clear\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_nodded_her_mind_racing\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bnodded,\\\\\\\\ her\\\\\\\\ mind\\\\\\\\ racing\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_finally_after_what_felt_like\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bFinally,\\\\\\\\ after\\\\\\\\ what\\\\\\\\ felt\\\\\\\\ like\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_something_far_more_sinister\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bsomething\\\\\\\\ far\\\\\\\\ more\\\\\\\\ sinister\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_air_was_thick_with_the_smell\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bair\\\\\\\\ was\\\\\\\\ thick\\\\\\\\ with\\\\\\\\ the\\\\\\\\ smell\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_eyes_snapped_open\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\beyes\\\\\\\\ snapped\\\\\\\\ open\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_like_polished_obsidian\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\blike\\\\\\\\ polished\\\\\\\\ obsidian\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_couldn_t_help_but_think\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bcouldn\'t\\\\\\\\ help\\\\\\\\ but\\\\\\\\ think\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_faint_smile_playing\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bfaint\\\\\\\\ smile\\\\\\\\ playing\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_carved_wooden_box\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bcarved\\\\\\\\ wooden\\\\\\\\ box\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_stumbled_upon_something\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bstumbled\\\\\\\\ upon\\\\\\\\ something\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_mixture_of_excitement_and_trepidation\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bmixture\\\\\\\\ of\\\\\\\\ excitement\\\\\\\\ and\\\\\\\\ trepidation\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_help_but_wonder_what_the_future\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bhelp\\\\\\\\ but\\\\\\\\ wonder\\\\\\\\ what\\\\\\\\ the\\\\\\\\ future\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_felt_a_profound_sense\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bfelt\\\\\\\\ a\\\\\\\\ profound\\\\\\\\ sense\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_phrase_casting_a_golden_hue\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"(?i)\\\\\\\\bcasting\\\\\\\\ a\\\\\\\\ golden\\\\\\\\ hue\\\\\\\\b\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_not_x_but\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"\\\\\\"(?i)not [^.!?]{3,60} but\\\\\\",\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_each_a\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"\\\\\\"(?i)each(?:\\\\\\\\s*\\\\\\\\w+\\\\\\\\s*|\\\\\\\\s*)a\\\\\\",\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n\\n- id: slop_every_a\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\"\\\\\\"(?i)every(?:\\\\\\\\s*\\\\\\\\w+\\\\\\\\s*|\\\\\\\\s*)a\\\\\\"\\"\\n  fix: \\"Known LLM overuse tell \\\\u2014 cut or rewrite plainly.\\"\\n  source: sam-paech/antislop-sampler\\n  license: Apache-2.0\\n  min_len_words: 0\\n  paper: paech-2510.15061\\n  enabled: true\\n", "scripts/fine_tune_lfm.py": "\\"\\"\\"Fine-tune an LFM2 backbone into the ITAIS detector.\\n\\nTwo paths, one script. The encoder path is the recommended default (see docs/HANDOFF.md \\u00a73):\\n\\n    # recommended: bidirectional encoder, doc head + per-token lane head, one body\\n    uv run python scripts/fine_tune_lfm.py --arch encoder --model LiquidAI/LFM2.5-Encoder-230M\\n\\n    # rev-1 alternative: causal decoder + LoRA sequence classifier, doc verdict only\\n    uv run python scripts/fine_tune_lfm.py --arch decoder --model LiquidAI/LFM2.5-1.2B --max-len 2048\\n\\n    # no corpus needed: synthetic data, 3 steps, proves the graph/loss/export path works\\n    uv run python scripts/fine_tune_lfm.py --arch encoder --smoke\\n\\nOutputs a bundle under --out: weights, tokenizer, calibration.json (threshold per register at 1% FPR)\\nand manifest.json (args, git sha, seed, per-slice metrics, trained_on / never_trained_on).\\n\\nNever emits or stores a \\"% AI\\" number: the doc head is a pile-resemblance score, calibrated against a\\nnamed human reference slice. See docs/HANDOFF.md \\u00a71.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nimport math\\nimport os\\nimport sys\\nimport random\\nimport subprocess\\nimport time\\nfrom dataclasses import asdict, dataclass\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport torch\\nfrom torch import nn\\n\\n# Ensure `slopdet` (src/) is importable no matter the CWD. Colab runs this as\\n# `python scripts/fine_tune_lfm.py` from a Drive dir; uv runs it from the repo root.\\n_ROOT = Path(__file__).resolve().parents[1]\\nif str(_ROOT / \\"src\\") not in sys.path:\\n    sys.path.insert(0, str(_ROOT / \\"src\\"))\\nif str(_ROOT / \\"scripts\\") not in sys.path:\\n    sys.path.insert(0, str(_ROOT / \\"scripts\\"))\\nfrom torch.utils.data import DataLoader, Dataset\\nfrom transformers import AutoTokenizer\\n\\nfrom slopdet.labels import parse_label\\nfrom slopdet.lfm import load_encoder_body\\n\\nDOC_LABELS = {\\"human\\": 0, \\"ai\\": 1}\\n\\n\\n@dataclass\\nclass Config:\\n    arch: str = \\"encoder\\"\\n    model: str = \\"LiquidAI/LFM2.5-Encoder-350M\\"\\n    max_len: int = 512\\n    batch_size: int = 8\\n    grad_accum: int = 4\\n    lr: float = 2e-5\\n    epochs: int = 1\\n    seed: int = 0\\n    token_loss_weight: float = 0.5\\n    precision: str = \\"fp16\\"\\n    ckpt_every: int = 500\\n\\n\\ndef set_seed(seed: int) -> None:\\n    random.seed(seed)\\n    np.random.seed(seed)\\n    torch.manual_seed(seed)\\n    torch.cuda.manual_seed_all(seed)\\n\\n\\ndef git_sha() -> str:\\n    try:\\n        return subprocess.check_output([\\"git\\", \\"rev-parse\\", \\"HEAD\\"], text=True).strip()\\n    except Exception:  # noqa: BLE001\\n        return \\"unknown\\"\\n\\n\\n# ---------------------------------------------------------------- data\\n\\n\\ndef load_rows(doc_parquet: Path | None, spans_parquet: Path | None, smoke: bool) -> list[dict]:\\n    \\"\\"\\"Rows are dicts: text, label (0/1), spans (list of {lane,start,end}), register.\\"\\"\\"\\n    if smoke:\\n        slop = \\"Here\'s the thing: we leverage robust pipelines to unlock synergies. \\"\\n        human = \\"Thursday mornings at the clinic were empty, so I counted 41 chairs. \\"\\n        rows = []\\n        for i in range(24):\\n            rows.append({\\"text\\": slop * 3, \\"label\\": 1, \\"register\\": \\"smoke\\",\\n                         \\"spans\\": [{\\"lane\\": \\"glue\\", \\"start\\": 25, \\"end\\": 33}]})\\n            rows.append({\\"text\\": human * 3, \\"label\\": 0, \\"register\\": \\"smoke\\", \\"spans\\": []})\\n        return rows\\n\\n    import pandas as pd\\n\\n    def _coerce_spans(spans) -> list[dict]:\\n        \\"\\"\\"Return the span dicts that carry a lane, tolerating None/str/list/ndarray.\\"\\"\\"\\n        if spans is None:\\n            return []\\n        if isinstance(spans, str):\\n            try:\\n                spans = json.loads(spans)\\n            except json.JSONDecodeError:\\n                return []\\n        if isinstance(spans, (list, tuple)):\\n            return [s for s in spans if isinstance(s, dict) and s.get(\\"lane\\")]\\n        # numpy array / Arrow list \\u2014 coerce safely\\n        try:\\n            return [s for s in spans if isinstance(s, dict) and s.get(\\"lane\\")]\\n        except TypeError:\\n            return []\\n\\n    def _read_chunked(path: Path, cols: list[str], chunk: int = 10_000):\\n        \\"\\"\\"Stream a parquet in slices so peak RAM stays bounded on Colab.\\"\\"\\"\\n        import pyarrow.parquet as pq\\n\\n        pf = pq.ParquetFile(path)\\n        for batch in pf.iter_batches(batch_size=chunk, columns=cols):\\n            yield batch.to_pandas()\\n\\n    if spans_parquet and spans_parquet.exists():\\n        import pyarrow.parquet as pq\\n\\n        cols = [c for c in (\\"text\\", \\"label\\", \\"pile\\", \\"register\\", \\"spans\\")\\n                if c in {f.name for f in pq.ParquetFile(spans_parquet).schema}]\\n        rows = []\\n        for chunk_df in _read_chunked(spans_parquet, cols):\\n            for rec in chunk_df.to_dict(\\"records\\"):\\n                rows.append({\\n                    \\"text\\": rec[\\"text\\"],\\n                    \\"label\\": parse_label(rec, default=0),\\n                    \\"register\\": rec.get(\\"register\\", \\"coai\\"),\\n                    \\"spans\\": _coerce_spans(rec.get(\\"spans\\")),\\n                })\\n        return rows\\n\\n    if not doc_parquet or not doc_parquet.exists():\\n        raise SystemExit(\\n            f\\"no corpus at {doc_parquet} / {spans_parquet}. Rebuild it with the commands in \\"\\n            \\"docs/HANDOFF.md \\u00a74, or pass --smoke to validate the code path without data.\\"\\n        )\\n    rows = []\\n    for chunk_df in _read_chunked(doc_parquet, [\\"text\\", \\"label\\", \\"register\\"]):\\n        for rec in chunk_df.to_dict(\\"records\\"):\\n            rows.append({\\"text\\": rec[\\"text\\"], \\"label\\": int(rec[\\"label\\"]),\\n                         \\"register\\": rec.get(\\"register\\", \\"coai\\"), \\"spans\\": []})\\n    return rows\\n\\n\\nclass SlopDataset(Dataset):\\n    def __init__(self, rows: list[dict], tok, max_len: int, lanes: list[str]):\\n        self.rows, self.tok, self.max_len = rows, tok, max_len\\n        self.lane_ids = {lane: i + 1 for i, lane in enumerate(lanes)}  # 0 = no lane\\n\\n    def __len__(self) -> int:\\n        return len(self.rows)\\n\\n    def __getitem__(self, idx: int) -> dict:\\n        row = self.rows[idx]\\n        enc = self.tok(row[\\"text\\"], truncation=True, max_length=self.max_len,\\n                       padding=\\"max_length\\", return_offsets_mapping=True)\\n        offsets = enc.pop(\\"offset_mapping\\")\\n        token_labels = [0] * len(offsets)\\n        for span in row[\\"spans\\"]:\\n            lane_id = self.lane_ids.get(span[\\"lane\\"])\\n            if not lane_id:\\n                continue\\n            for pos, (start, end) in enumerate(offsets):\\n                if end > start and start >= span[\\"start\\"] and end <= span[\\"end\\"]:\\n                    token_labels[pos] = lane_id\\n        item = {k: torch.tensor(v) for k, v in enc.items()}\\n        item[\\"doc_label\\"] = torch.tensor(row[\\"label\\"])\\n        item[\\"token_labels\\"] = torch.tensor(token_labels)\\n        return item\\n\\n\\n# ---------------------------------------------------------------- model\\n\\n\\nclass EncoderDetector(nn.Module):\\n    \\"\\"\\"LFM2 bidirectional body, mean-pooled doc head + per-token lane head.\\"\\"\\"\\n\\n    def __init__(self, model_name: str, n_lanes: int):\\n        super().__init__()\\n        self.body = load_encoder_body(model_name)\\n        hidden = self.body.config.hidden_size\\n        self.doc = nn.Linear(hidden, 2)\\n        self.token = nn.Linear(hidden, n_lanes + 1)\\n\\n    def forward(self, input_ids, attention_mask):\\n        states = self.body(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state\\n        mask = attention_mask.unsqueeze(-1).to(states.dtype)\\n        pooled = (states * mask).sum(1) / mask.sum(1).clamp(min=1e-6)\\n        return self.doc(pooled), self.token(states)\\n\\n\\ndef build_decoder(model_name: str):\\n    \\"\\"\\"Rev-1 path: causal LM with a 2-class head over the last token, LoRA-adapted.\\"\\"\\"\\n    from peft import LoraConfig, get_peft_model\\n    from transformers import AutoModelForSequenceClassification\\n\\n    model = AutoModelForSequenceClassification.from_pretrained(\\n        model_name, num_labels=2, trust_remote_code=True)\\n    model.config.use_cache = False\\n    if model.config.pad_token_id is None:\\n        model.config.pad_token_id = model.config.eos_token_id\\n    return get_peft_model(model, LoraConfig(\\n        r=16, lora_alpha=32, lora_dropout=0.05, bias=\\"none\\",\\n        task_type=\\"SEQ_CLS\\", target_modules=\\"all-linear\\"))\\n\\n\\n# ---------------------------------------------------------------- metrics\\n\\n\\ndef auroc(scores: list[float], labels: list[int]) -> float:\\n    pairs = sorted(zip(scores, labels))\\n    pos = sum(labels)\\n    neg = len(labels) - pos\\n    if not pos or not neg:\\n        return float(\\"nan\\")\\n    rank_sum, rank = 0.0, 0\\n    while rank < len(pairs):\\n        tied = [i for i in range(rank, len(pairs)) if pairs[i][0] == pairs[rank][0]]\\n        avg_rank = sum(i + 1 for i in tied) / len(tied)\\n        rank_sum += sum(avg_rank for i in tied if pairs[i][1] == 1)\\n        rank += len(tied)\\n    return (rank_sum - pos * (pos + 1) / 2) / (pos * neg)\\n\\n\\ndef tpr_at_fpr(scores: list[float], labels: list[int], fpr: float = 0.01) -> tuple[float, float]:\\n    human = sorted((s for s, y in zip(scores, labels) if y == 0), reverse=True)\\n    if not human:\\n        return float(\\"nan\\"), float(\\"nan\\")\\n    threshold = human[min(int(len(human) * fpr), len(human) - 1)]\\n    ai = [s for s, y in zip(scores, labels) if y == 1]\\n    tpr = sum(s > threshold for s in ai) / len(ai) if ai else float(\\"nan\\")\\n    return tpr, threshold\\n\\n\\n# ---------------------------------------------------------------- train\\n\\n\\ndef evaluate(model, loader, device, arch: str) -> tuple[list[float], list[int]]:\\n    model.eval()\\n    scores, labels = [], []\\n    with torch.no_grad():\\n        for batch in loader:\\n            ids = batch[\\"input_ids\\"].to(device)\\n            mask = batch[\\"attention_mask\\"].to(device)\\n            doc_logits = model(ids, mask)[0] if arch == \\"encoder\\" else model(\\n                input_ids=ids, attention_mask=mask).logits\\n            scores += torch.softmax(doc_logits.float(), -1)[:, 1].cpu().tolist()\\n            labels += batch[\\"doc_label\\"].tolist()\\n    model.train()\\n    return scores, labels\\n\\n\\ndef main() -> int:\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\"--arch\\", choices=[\\"encoder\\", \\"decoder\\"], default=\\"encoder\\")\\n    ap.add_argument(\\"--model\\", default=Config.model)\\n    ap.add_argument(\\"--doc-parquet\\", type=Path, default=Path(\\"data/coai_train.parquet\\"))\\n    ap.add_argument(\\"--spans-parquet\\", type=Path, default=Path(\\"data/training/spans_coai_train.parquet\\"))\\n    ap.add_argument(\\"--out\\", type=Path, default=Path(\\"artifacts/lfm\\"))\\n    ap.add_argument(\\"--max-len\\", type=int, default=Config.max_len)\\n    ap.add_argument(\\"--batch-size\\", type=int, default=Config.batch_size)\\n    ap.add_argument(\\"--grad-accum\\", type=int, default=Config.grad_accum)\\n    ap.add_argument(\\"--lr\\", type=float, default=Config.lr)\\n    ap.add_argument(\\"--epochs\\", type=int, default=Config.epochs)\\n    ap.add_argument(\\"--seed\\", type=int, default=Config.seed)\\n    ap.add_argument(\\"--precision\\", choices=[\\"fp16\\", \\"fp32\\"], default=Config.precision)\\n    ap.add_argument(\\"--ckpt-every\\", type=int, default=Config.ckpt_every)\\n    ap.add_argument(\\"--val-frac\\", type=float, default=0.05)\\n    ap.add_argument(\\"--smoke\\", action=\\"store_true\\", help=\\"synthetic data, 3 steps, no corpus needed\\")\\n    args = ap.parse_args()\\n\\n    cfg = Config(arch=args.arch, model=args.model, max_len=args.max_len, batch_size=args.batch_size,\\n                 grad_accum=args.grad_accum, lr=args.lr, epochs=args.epochs, seed=args.seed,\\n                 precision=args.precision, ckpt_every=args.ckpt_every)\\n    set_seed(cfg.seed)\\n    device = torch.device(\\"cuda\\" if torch.cuda.is_available() else \\"cpu\\")\\n    if cfg.precision == \\"fp16\\" and device.type == \\"cuda\\" and torch.cuda.get_device_capability()[0] < 8:\\n        print(\\"[warn] Turing-class GPU: no bf16 and no flash-attn 2. fp16 + NaN preflight it is.\\")\\n\\n    rows = load_rows(args.doc_parquet, args.spans_parquet, args.smoke)\\n    lanes = sorted({s[\\"lane\\"] for r in rows for s in r[\\"spans\\"]})\\n    random.shuffle(rows)\\n    val_rows, train_rows = [], []\\n    for register in {r[\\"register\\"] for r in rows}:\\n        for label in (0, 1):\\n            group = [r for r in rows if r[\\"register\\"] == register and r[\\"label\\"] == label]\\n            take = max(1, int(len(group) * args.val_frac)) if group else 0\\n            val_rows += group[:take]\\n            train_rows += group[take:]\\n    random.shuffle(train_rows)\\n    print(f\\"[data] {len(train_rows)} train / {len(val_rows)} val \\u00b7 {len(lanes)} lanes\\")\\n\\n    tok = AutoTokenizer.from_pretrained(cfg.model, trust_remote_code=True)\\n    train_ds = SlopDataset(train_rows, tok, cfg.max_len, lanes)\\n    val_ds = SlopDataset(val_rows, tok, cfg.max_len, lanes)\\n    train_dl = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, drop_last=False)\\n    val_dl = DataLoader(val_ds, batch_size=cfg.batch_size)\\n\\n    model = (EncoderDetector(cfg.model, len(lanes)) if cfg.arch == \\"encoder\\"\\n             else build_decoder(cfg.model)).to(device)\\n    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=0.01)\\n    use_amp = cfg.precision == \\"fp16\\" and device.type == \\"cuda\\"\\n    scaler = torch.amp.GradScaler(\\"cuda\\", enabled=use_amp)\\n    doc_loss_fn = nn.CrossEntropyLoss()\\n    # K4 fix: class-balanced token loss. >99% of tokens are \\"no lane\\" (class 0),\\n    # so an unweighted token CE collapses to \\"no lane everywhere\\". Weight each\\n    # lane class inversely to its frequency so the head actually learns spans.\\n    lane_ids = {lane: i + 1 for i, lane in enumerate(lanes)}  # 0 = no lane\\n    token_weights = torch.ones(len(lanes) + 1, device=device)  # +1 for class 0\\n    lane_counts = torch.zeros(len(lanes) + 1, device=device)\\n    for r in rows:\\n        for s in r[\\"spans\\"]:\\n            lid = lane_ids.get(s[\\"lane\\"], 0)\\n            lane_counts[lid] += max(0, (s[\\"end\\"] - s[\\"start\\"]))\\n    total = lane_counts.sum().clamp(min=1)\\n    # inverse-frequency, capped so rare lanes don\'t explode\\n    token_weights[1:] = (total / lane_counts[1:].clamp(min=1)).clamp(max=50)\\n    print(f\\"[data] token class weights: {token_weights.tolist()}\\", flush=True)\\n    token_loss_fn = nn.CrossEntropyLoss(ignore_index=-100, weight=token_weights)\\n\\n    total_steps = max(1, len(train_dl) // cfg.grad_accum) * cfg.epochs\\n    max_steps = 3 if args.smoke else total_steps * cfg.grad_accum\\n    args.out.mkdir(parents=True, exist_ok=True)\\n    step, started = 0, time.time()\\n\\n    for epoch in range(cfg.epochs):\\n        for batch in train_dl:\\n            ids = batch[\\"input_ids\\"].to(device)\\n            mask = batch[\\"attention_mask\\"].to(device)\\n            doc_y = batch[\\"doc_label\\"].to(device)\\n            with torch.autocast(\\"cuda\\", dtype=torch.float16, enabled=use_amp):\\n                if cfg.arch == \\"encoder\\":\\n                    doc_logits, token_logits = model(ids, mask)\\n                    token_y = batch[\\"token_labels\\"].to(device).masked_fill(mask == 0, -100)\\n                    loss = doc_loss_fn(doc_logits, doc_y) + cfg.token_loss_weight * token_loss_fn(\\n                        token_logits.reshape(-1, token_logits.size(-1)), token_y.reshape(-1))\\n                else:\\n                    loss = doc_loss_fn(model(input_ids=ids, attention_mask=mask).logits, doc_y)\\n            if not torch.isfinite(loss):\\n                raise SystemExit(\\n                    \\"[abort] non-finite loss. On Turing (T4) fp16 is the usual cause: rerun with \\"\\n                    \\"--precision fp32, or lower --lr. Do not \'fix\' this by filtering the log.\\"\\n                )\\n            scaler.scale(loss / cfg.grad_accum).backward()\\n            step += 1\\n            if step % cfg.grad_accum == 0:\\n                scaler.unscale_(opt)\\n                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\\n                scaler.step(opt)\\n                scaler.update()\\n                opt.zero_grad(set_to_none=True)\\n            if step % 50 == 0 or args.smoke:\\n                print(f\\"[train] epoch {epoch} step {step}/{max_steps} loss {loss.item():.4f}\\")\\n            if cfg.ckpt_every and step % cfg.ckpt_every == 0:\\n                torch.save({\\"step\\": step, \\"model\\": model.state_dict()}, args.out / \\"checkpoint.pt\\")\\n            if step >= max_steps:\\n                break\\n        if step >= max_steps:\\n            break\\n\\n    scores, labels = evaluate(model, val_dl, device, cfg.arch)\\n    registers = [r[\\"register\\"] for r in val_rows]\\n    metrics: dict[str, dict] = {}\\n    for register in sorted(set(registers)):\\n        idx = [i for i, r in enumerate(registers) if r == register]\\n        s = [scores[i] for i in idx]\\n        y = [labels[i] for i in idx]\\n        tpr, threshold = tpr_at_fpr(s, y)\\n        metrics[register] = {\\"n\\": len(idx), \\"auroc\\": auroc(s, y), \\"tpr_at_1pct_fpr\\": tpr,\\n                             \\"threshold\\": threshold}\\n    tpr_all, threshold_all = tpr_at_fpr(scores, labels)\\n    metrics[\\"all\\"] = {\\"n\\": len(labels), \\"auroc\\": auroc(scores, labels),\\n                      \\"tpr_at_1pct_fpr\\": tpr_all, \\"threshold\\": threshold_all}\\n\\n    torch.save(model.state_dict(), args.out / \\"model.pt\\")\\n    tok.save_pretrained(args.out)\\n    (args.out / \\"calibration.json\\").write_text(json.dumps(\\n        {\\"fpr\\": 0.01, \\"per_register\\": {k: v[\\"threshold\\"] for k, v in metrics.items()}}, indent=2) + \\"\\\\n\\")\\n    (args.out / \\"manifest.json\\").write_text(json.dumps({\\n        \\"config\\": asdict(cfg), \\"lanes\\": lanes, \\"git_sha\\": git_sha(), \\"metrics\\": metrics,\\n        \\"trained_on\\": [str(args.spans_parquet if args.spans_parquet.exists() else args.doc_parquet)]\\n        if not args.smoke else [\\"synthetic-smoke\\"],\\n        \\"never_trained_on\\": [\\"eval/labels/laguna.jsonl\\", \\"eval/labels/local.jsonl\\"],\\n        \\"wall_seconds\\": round(time.time() - started, 1),\\n        \\"note\\": \\"doc head is pile resemblance, not an authorship or %-AI claim\\",\\n    }, indent=2) + \\"\\\\n\\")\\n    print(json.dumps(metrics, indent=2))\\n    print(f\\"[done] bundle at {args.out}\\")\\n    return 0\\n\\n\\nif __name__ == \\"__main__\\":\\n    raise SystemExit(main())\\n"}')
for rel, content in FILES.items():
    path = Path(rel)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")
print("wrote", len(FILES), "files")
import sys
sys.path.insert(0, str(Path("src").resolve()))
print("ITAIS ready")


In [ ]:
# Locate the training parquet: Drive → /content → manual upload
from pathlib import Path

DATA_PARQUET = "train_all.parquet"
candidates = [
    Path(".") / DATA_PARQUET,                     # already in ROOT
    Path("/content") / DATA_PARQUET,              # uploaded to /content
    Path("/content/drive/MyDrive/isthisaislop") / DATA_PARQUET,
]
src = next((p for p in candidates if p.exists()), None)
if src is None:
    raise SystemExit(
        f"train_all.parquet not found. Upload it to Drive {DRIVE_PATH}/ or /content/, "
        f"or re-run the build locally and copy it."
    )
if src.resolve() != (Path(".") / DATA_PARQUET).resolve():
    import shutil
    shutil.copy(src, Path(".") / DATA_PARQUET)
    print(f"copied {src} → ./{DATA_PARQUET}")
else:
    print(f"using ./{DATA_PARQUET}")


In [ ]:
# DIAGNOSTIC — run the trainer inline so the real exception shows
# If the training cell fails with rc=1 and no message, run THIS cell:
import sys, traceback
sys.path.insert(0, str(Path("scripts").resolve()))
sys.path.insert(0, str(Path("src").resolve()))
import fine_tune_lfm
try:
    sys.argv = ["fine_tune_lfm.py", "--arch", "encoder", "--model", MODEL,
                "--spans-parquet", DATA_PARQUET, "--max-len", str(MAX_LEN),
                "--epochs", str(EPOCHS), "--out", "artifacts/lfm",
                "--precision", "fp32"]
    rc = fine_tune_lfm.main()
    print("main() returned", rc)
except SystemExit as e:
    print("SystemExit:", e)
except Exception:
    traceback.print_exc()
print("DIAGNOSTIC DONE")


In [ ]:
# Train: LFM2.5-Encoder-350M, 1 epoch; fp16 with automatic fp32 fallback
import sys, subprocess
from pathlib import Path

def run_training(extra: list[str]) -> int:
    cmd = [
        sys.executable, "scripts/fine_tune_lfm.py",
        "--arch", "encoder",
        "--model", MODEL,
        "--spans-parquet", DATA_PARQUET,
        "--max-len", str(MAX_LEN),
        "--epochs", str(EPOCHS),
        "--out", "artifacts/lfm",
    ] + extra
    print("running:", " ".join(cmd), flush=True)
    # Stream the child's output live so the real error is visible,
    # and capture it so we can print the tail on failure.
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    lines = []
    for line in proc.stdout:
        lines.append(line)
        print(line, end="", flush=True)
    rc = proc.wait()
    if rc != 0:
        print("\n[FAILED rc=%d] last 30 lines:" % rc, flush=True)
        for l in lines[-30:]:
            print("   | " + l.rstrip(), flush=True)
    return rc

# Try fp16 first (default). On Turing T4, fp16 can NaN-abort; fall back to fp32.
rc = run_training([])
if rc != 0:
    print("\n[retry] fp16 run failed (rc=%d). Retrying with --precision fp32..." % rc, flush=True)
    rc = run_training(["--precision", "fp32"])
sys.exit(rc)
